# Chapter 12: Monetary Policy and the Phillips Curve

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quarcs-lab/macro102-site/blob/main/notebooks/ch12-monetary-policy-and-the-phillips-curve.ipynb)

*Data Science and Macroeconomic Models: A Python Companion to Intermediate Macroeconomics* · Carlos Mendez

> **How to use this notebook.** In Colab, click *Runtime → Run all*. After that, every model
> figure has live sliders: move one and the model is solved again on the spot. On the book
> website the same figures come with one precomputed slider. Each section ends with a short
> **Your turn** task, and you can try it in the empty code cell below the task.

## Learning goals

Chapter 11 built the IS curve, which links the real interest rate to short-run output. This chapter adds the
two missing pieces of the short-run model: the **MP curve** (how the central bank sets the real interest rate)
and the **Phillips curve** (how output moves inflation over time). By the end of it you will be able to:

- explain why a central bank that sets a **nominal** interest rate also sets the **real** interest rate in the
  short run, using the Fisher equation and the assumption of **sticky inflation**;
- use the **IS/MP diagram** to analyze monetary policy and demand shocks, such as the end of a housing bubble;
- explain the **Phillips curve** $\Delta\pi_t = \bar{\nu}\tilde{Y}_t + \bar{o}$, with **adaptive expectations** and **price shocks**;
- simulate the **Volcker disinflation** year by year and explain how mistakes contributed to the **Great Inflation**;
- read U.S. data on the federal funds rate, the yield curve, inflation and the Covid-19 recession;
- in Python: draw several panels with `make_subplots`, simulate a model with a loop, reshape tables with
  `melt` and `pivot`, animate a chart with `animation_frame`, and fit a trend with `np.polyfit`.

## Figures in this chapter

The figures follow the order of the textbook chapter. The second column shows which textbook figure each
notebook figure replicates. Figure 12.1 is a diagram of the model's structure; here it becomes a small live
version of the model. Figure 12.21 is the textbook's worked answer to Exercise 12.

| Figure | Textbook figure | Type | What you can change |
|---|---|---|---|
| Figure 1 | 12.1 The structure of the short-run model | model | nominal interest rate, sticky inflation, demand shock |
| Figure 2 | 12.2 The federal funds rate | data | the period shown (menu) |
| Figure 3 | 12.3 The MP curve in the IS/MP diagram | model | real interest rate set by the central bank, demand shock ā, slope b̄ |
| Figure 4 | 12.4 Raising the interest rate in the IS/MP diagram | model | new real interest rate R′ (and ā′, b̄) |
| Figure 5 | 12.5 Stabilizing the economy after a housing bubble | model | size of the demand shock ā′, slope b̄ |
| Figure 6 | 12.6 The yield curve for U.S. Treasuries | data | play the yield curve month by month (animation) |
| Figure 7 | 12.7 The Phillips curve | model | short-run output Ỹ, price shock ō, slope ν̄ |
| Figure 8 | 12.8 An oil price increase | model | price shock ō (and Ỹ, ν̄) |
| Figure 9 | 12.9 Inflation in the United States | data | show the fed funds rate and the real interest rate (legend) |
| Figure 10 | 12.10 Tightening monetary policy | model | new nominal interest rate i′, sticky inflation π |
| Figure 11 | 12.11 A recession and falling inflation | model | short-run output in the recession Ỹ′ (and ō, ν̄) |
| Figure 12 | 12.12 The disinflation over time | model | depth of the recession, years of tight policy t\*, slope ν̄; "Run the Volcker policy" button |
| Figure 13 | 12.13 Mistaking a slowdown in potential for a recession | model | potential growth before and after 1973, size of the stimulus |
| Figure 14 | 12.14 Real GDP in the United States | data | hover; the pre-Covid trend (legend) |
| Figure 15 | 12.15 Consumer price index inflation in the United States | data | hover; all items and core (legend) |
| Figure 16 | 12.16 U.S. federal government spending as a share of GDP | data | hover |
| Figure 17 | 12.17 Income and consumption | data | hover; the series shown (legend) |
| Figure 18 | 12.18 How the central bank sets the nominal interest rate | model | money supply Mˢ, money demand |
| Figure 19 | 12.19 Raising the nominal interest rate | model | new money supply Mˢ′ (and money demand) |
| Figure 20 | 12.20 Targeting the nominal interest rate | model | money-demand shock, target rate i\*, policy (peg the rate or fix the money supply) |
| Figure 21 | 12.21 The Great Inflation | model | the demand shock policymakers believe in, slope ν̄ |

## Setup

We first load the Python libraries we need. You only have to run this cell once.

> **Python note.** `import` loads a *library*, a toolbox written by other people. We give each
> library a short nickname, for example `np` for NumPy, so that the code stays short. Text after
> a `#` is a *comment*: Python ignores it, and it is there for human readers. The `try ... except`
> block runs one Colab-only line and simply skips it anywhere else.

In [1]:
# Libraries used in this chapter
import numpy as np                              # numbers and arrays
import pandas as pd                             # data tables
import plotly.express as px                     # quick charts from data tables
import plotly.graph_objects as go               # charts built piece by piece (model figures)
import plotly.io as pio
from plotly.subplots import make_subplots       # several panels in one figure
from ipywidgets import interact, interact_manual, FloatSlider, IntSlider, ToggleButtons, Dropdown

# One light blue/green look for every chart
BLUE, TEAL, GREEN, GRAY = "#16528c", "#008070", "#7fb23a", "#8a96a3"
pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.update(width=860, height=480)   # same size in Colab and on the website
px.defaults.color_discrete_sequence = [BLUE, TEAL, GREEN, "#5aa0d8", "#0b4f47"]
WIDE = {"description_width": "initial"}         # show the full slider labels

# Colab only: let Plotly charts appear inside the interactive widgets
try:
    import google.colab
    google.colab.output.enable_custom_widget_manager()
except ImportError:
    pass                                        # not running in Colab: nothing to do

# The data files of this chapter live in the book's GitHub repository
DATA_URL = "https://raw.githubusercontent.com/quarcs-lab/macro102-site/main/data/ch12/"

The next cell defines a helper that only the book website needs. Run it and move on.

In [2]:
# @title Helper for the website versions (run this cell; you do not need to read it)
def slider_figure(make_figure, name, values, start, label=None):
    """Precompute make_figure(name=value) for every value and join the results with one slider.

    The website cannot run Python, so each interactive figure also gets this precomputed version.
    """
    figures = [make_figure(**{name: v}) for v in values]
    first = values.index(start)
    fig = go.Figure(layout=figures[first].layout)
    owner = []                                  # which precomputed figure each trace comes from
    for i, f in enumerate(figures):
        for trace in f.data:
            trace.visible = (i == first)
            fig.add_trace(trace)
            owner.append(i)
    steps = []
    for i, f in enumerate(figures):
        layout = {"title": f.layout.title.to_plotly_json(),
                  "annotations": [a.to_plotly_json() for a in f.layout.annotations],
                  "shapes": [s.to_plotly_json() for s in f.layout.shapes]}
        visible = [o == i for o in owner]
        steps.append(dict(method="update", label=str(values[i]), args=[{"visible": visible}, layout]))
    fig.update_layout(sliders=[dict(active=first, steps=steps, pad=dict(t=60),
                                    currentvalue=dict(prefix=(label or name) + " = "))])
    return fig

## The model in code

The short-run model has three building blocks. Each one links two variables, and together they form a chain
from the central bank's interest rate to inflation.

**1. The MP curve.** The central bank sets a short-term **nominal** interest rate $i_t$. The **Fisher equation**
links it to the **real** interest rate $R_t$ and the inflation rate $\pi_t$:

$$i_t = R_t + \pi_t \qquad (12.1) \qquad\Longleftrightarrow\qquad R_t = i_t - \pi_t. \qquad (12.2)$$

The key assumption is **sticky inflation**: in the very short run (say six months), inflation does not react to
monetary policy. So when the central bank moves $i_t$, $R_t$ moves one for one. In effect the central bank
chooses the real interest rate, and the **MP curve** is a horizontal line at that rate.

**2. The IS curve** of Chapter 11 turns the real interest rate into short-run output:

$$\tilde{Y}_t = \bar{a} - \bar{b}\,(R_t - \bar{r}). \qquad (11.12)$$

**3. The Phillips curve.** Firms raise their prices by the inflation they expect, plus an adjustment for demand.
With **adaptive expectations**, $\pi^e_t = \pi_{t-1}$, and with a **price shock** $\bar{o}$ (an oil shock, for example):

$$\pi_t = \pi_{t-1} + \bar{\nu}\tilde{Y}_t + \bar{o} \qquad (12.7) \qquad\Longleftrightarrow\qquad \Delta\pi_t = \bar{\nu}\tilde{Y}_t + \bar{o}. \qquad (12.8)$$

**One baseline for every diagram.** The textbook draws its diagrams without numbers, apart from a few
examples. All the figures of this notebook start from one set of values:

- the IS curve of Chapter 11's numerical example: no demand shock $\bar{a} = 0$, $\bar{b} = 2$ and a marginal
  product of capital $\bar{r} = 5$ percent. Then the textbook's "$-2$ percent" in Figures 12.4, 12.5 and 12.10
  comes from $R' = 6$ percent or from $\bar{a}' = -2$;
- the slope of the Phillips curve $\bar{\nu} = 0.5$ (illustrative; Chapter 9 estimated about 1/3 for the United
  States) and no price shock, $\bar{o} = 0$;
- sticky inflation of 2 percent in normal times and 10 percent at the start of the Volcker disinflation (the
  textbook's number);
- a money market with money demand $M^d = 10 - i$ and money supply $\bar{M}^s = 5$ (illustrative units), so that
  the nominal interest rate is 5 percent;
- for the productivity slowdown of Figure 13, potential output growth of 3.5 percent a year before 1973 and
  2.5 percent after (illustrative).

> **Python note.** `def` creates a *function*: a named recipe that takes inputs (the *arguments*) and gives
> back a result with `return`. Writing `b=B_BAR` gives the argument a *default value*, used whenever we do
> not pass one; here the default is a constant we defined just before. The text in triple quotes is a short
> description of what the function does.

In [3]:
A_BAR, B_BAR, R_BAR = 0.0, 2.0, 5.0     # IS curve: no demand shock, b = 2, r_bar = 5 percent
NU, O_BAR = 0.5, 0.0                     # Phillips curve: slope nu = 0.5, no price shock
M0, M_BAR = 10.0, 5.0                    # money market: demand M = 10 - i, supply 5 (units)


def real_rate(i, pi):
    """Equation 12.2, the Fisher equation: the real interest rate R = i - pi, in percent."""
    return i - pi


def is_curve(R, a=A_BAR, b=B_BAR, r_bar=R_BAR):
    """Equation 11.12, the IS curve: short-run output Y~ = a - b (R - r_bar), in percent."""
    return a - b * (R - r_bar)


def rate_for_output(Y, a=A_BAR, b=B_BAR, r_bar=R_BAR):
    """The IS curve solved for R: the real interest rate at which short-run output equals Y."""
    return r_bar + (a - Y) / b


def phillips(Y, nu=NU, o=O_BAR):
    """Equation 12.8, the Phillips curve: the change in inflation = nu * Y~ + o, in percentage points."""
    return nu * Y + o

The money market of Section 12.7 has two more functions. People hold less money when the nominal interest
rate, the opportunity cost of holding money, is higher.

In [4]:
def money_demand(i, m0=M0):
    """Figure 12.18: money demanded falls as the nominal interest rate rises, M = m0 - i."""
    return m0 - i


def market_rate(M_s, m0=M0):
    """The nominal rate that clears the money market: m0 - i = M_s, so i = m0 - M_s."""
    return m0 - M_s

The full model runs year by year: the central bank sets $R_t$, the IS curve gives $\tilde{Y}_t$, and the Phillips
curve turns last year's inflation into this year's. `simulate` repeats these three steps for a whole path of
interest rates.

> **Python note.** A `for` loop repeats the indented lines once for each number in `R_path`. `Y_path = []`
> starts an empty list, and `.append(...)` adds one number at its end. `np.array(...)` turns a list into an
> array, so that we can do arithmetic with it.

In [5]:
def simulate(R_path, pi_start, nu=NU, a=A_BAR):
    """The short-run model year by year: MP sets R, IS gives Y~, the Phillips curve gives pi (eq. 12.7)."""
    Y_path, pi_path = [], []
    pi = pi_start                          # inflation of the year before the first year
    for R in R_path:
        Y = is_curve(R, a)
        pi = pi + phillips(Y, nu)          # last year's inflation, adjusted for demand
        Y_path.append(Y)
        pi_path.append(pi)
    return np.array(Y_path), np.array(pi_path)

Let us try the functions on the textbook's examples.

> **Python note.** An *f-string* such as `f"R = {value:.1f}%"` inserts the value of a variable into a piece of
> text; `:.1f` means "show one decimal" and `:+.2f` "show the sign and two decimals".

In [6]:
print(f"Fisher: i = 7%, pi = 2%    ->  R = {real_rate(7.0, 2.0):.1f}%")
print(f"IS: R = 5%  ->  Y~ = {is_curve(5.0):+.1f}%;   R = 6%  ->  Y~ = {is_curve(6.0):+.1f}%")
print(f"Housing bust a' = -2: the rate that keeps Y~ = 0 is R' = {rate_for_output(0.0, a=-2.0):.1f}%")
print(f"Phillips curve: Y~ = -2%  ->  change in inflation {phillips(-2.0):+.2f} points")
print(f"Oil shock o = 1 at Y~ = 0  ->  change in inflation {phillips(0.0, o=1.0):+.2f} points")
print(f"Money market: supply 5  ->  i = {market_rate(5.0):.1f}%;   supply 4  ->  i = {market_rate(4.0):.1f}%")
Y_path, pi_path = simulate([6.0, 6.0, 6.0, 6.0, 6.0, 5.0], pi_start=10.0)
print("Volcker: five years at R = 6%, then R = 5%. Inflation:", pi_path)

Fisher: i = 7%, pi = 2%    ->  R = 5.0%
IS: R = 5%  ->  Y~ = +0.0%;   R = 6%  ->  Y~ = -2.0%
Housing bust a' = -2: the rate that keeps Y~ = 0 is R' = 4.0%
Phillips curve: Y~ = -2%  ->  change in inflation -1.00 points
Oil shock o = 1 at Y~ = 0  ->  change in inflation +1.00 points
Money market: supply 5  ->  i = 5.0%;   supply 4  ->  i = 6.0%
Volcker: five years at R = 6%, then R = 5%. Inflation: [9. 8. 7. 6. 5. 5.]


Raising the real interest rate from 5 to 6 percent pushes output 2 percent below potential, as in the textbook.
With $\bar{\nu} = 0.5$, a 2 percent recession lowers inflation by 1 point a year, so five years of it bring inflation
from 10 down to 5 percent, and it stays there once the interest rate is back at $\bar{r}$.

### Drawing tools

All the diagrams of this chapter are built from the same few pieces, so we write them once:

- `add_line` draws a curve through some points and writes its label at the right end, as the textbook does;
- `add_is`, `add_pc` and `add_demand` draw an IS curve, a Phillips curve and a money demand curve inside
  the fixed axes;
- `add_point` marks a point with dashed guides to the axes, and `add_arrow` draws an arrow;
- `set_axes` sets fixed axes, so that you can see the curves move when you change a slider.

Every function takes `row` and `col` arguments, so it can draw in one panel of a figure with several panels.

> **Python note.** `go.Figure()` starts an empty chart and `add_trace` adds a line or a set of points. `[""] * 3`
> is the list `["", "", ""]`, and `+ [label]` adds one more element, so only the last point of a curve gets the
> label. `max(a, b)` and `min(a, b)` give the larger and the smaller of two numbers. `!=` means "is not equal to".

In [7]:
def add_line(fig, x, y, label, color, dash="solid", row=None, col=None):
    """Draw a curve through the points (x, y) and write its label at the right end."""
    text = [""] * (len(x) - 1) + [label]
    fig.add_trace(go.Scatter(x=x, y=y, mode="lines+text", text=text, textposition="middle right",
                             textfont=dict(size=15, color=color), cliponaxis=False, showlegend=False,
                             line=dict(color=color, width=4, dash=dash)), row=row, col=col)


def add_point(fig, x, y, label, x0, y0, row=None, col=None):
    """Mark the point (x, y) with a dot, a label and dashed guides to the axes, which start at (x0, y0)."""
    fig.add_trace(go.Scatter(x=[x0, x, x], y=[y, y, y0], mode="lines", showlegend=False, hoverinfo="skip",
                             line=dict(color="black", width=1, dash="dash")), row=row, col=col)
    fig.add_trace(go.Scatter(x=[x], y=[y], mode="markers+text", text=[label], textposition="top right",
                             textfont=dict(size=16), showlegend=False, marker=dict(color="black", size=12)),
                  row=row, col=col)


def add_arrow(fig, x0, y0, x1, y1, row=None, col=None):
    """Draw an arrow from (x0, y0) to (x1, y1); nothing if the two points are the same."""
    if (x0, y0) != (x1, y1):
        fig.add_trace(go.Scatter(x=[x0, x1], y=[y0, y1], mode="lines+markers", showlegend=False, hoverinfo="skip",
                                 line=dict(color="black", width=2),
                                 marker=dict(symbol="arrow", size=[0, 14], angleref="previous", color="black")),
                      row=row, col=col)

In [8]:
def add_is(fig, label, color, a=A_BAR, b=B_BAR, dash="solid", row=None, col=None):
    """Draw the IS curve inside the axes (R between 1 and 9.5 percent, Y~ between -7 and 6)."""
    Y = np.array([max(-7.0, is_curve(9.5, a, b)), min(6.0, is_curve(1.0, a, b))])
    add_line(fig, Y, rate_for_output(Y, a, b), label, color, dash, row, col)


def add_pc(fig, label, color, nu=NU, o=O_BAR, dash="solid", row=None, col=None):
    """Draw the Phillips curve inside the axes (change in inflation between -3.6 and 3.4 points)."""
    Y = np.array([max(-7.0, (-3.6 - o) / nu), min(6.0, (3.4 - o) / nu)])
    add_line(fig, Y, phillips(Y, nu, o), label, color, dash, row, col)


def add_demand(fig, label, color, m0=M0, dash="solid"):
    """Draw the money demand curve M = m0 - i for nominal rates from 9 down to 1 percent."""
    i = np.array([9.0, 1.0])
    add_line(fig, money_demand(i, m0), i, label, color, dash)


def set_axes(fig, xaxis, yaxis, row=None, col=None):
    """Fixed axes (title, range, ticks), so that you can see the curves move."""
    fig.update_xaxes(xaxis, row=row, col=col)
    fig.update_yaxes(yaxis, row=row, col=col)

The three kinds of diagram use these axes. The lowest tick of each axis is left out, so that no two numbers
crowd the bottom-left corner.

> **Python note.** `dict(title=..., range=...)` is a *dictionary*: a set of named settings. We store the
> settings of each axis once and hand them to `set_axes`.

In [9]:
X_OUTPUT = dict(title="Short-run output, Ỹ (%)", range=[-8, 8], tickvals=[-6, -4, -2, 0, 2, 4, 6], zeroline=False)
Y_RATE = dict(title="Real interest rate, R (%)", range=[0, 10], tickvals=[2, 4, 6, 8], zeroline=False)
Y_DPI = dict(title="Change in inflation, Δπ", range=[-4, 4], tickvals=[-3, -2, -1, 0, 1, 2, 3], zeroline=False)
X_MONEY = dict(title="Money, M", range=[0, 12], tickvals=[2, 4, 6, 8, 10], zeroline=False)
Y_NOMINAL = dict(title="Nominal interest rate, i (%)", range=[0, 10], tickvals=[2, 4, 6, 8], zeroline=False)
MP_X = [-8, 6.5]                         # the MP line runs across the diagram, leaving room for its label

### Shading recessions

Six data figures shade the U.S. recessions, as dated by the National Bureau of Economic Research (NBER). The
file `recessions.csv` lists, for each recession since 1953, the **peak** (the last month before the recession)
and the **trough** (its last month).

> **Python note.** `pd.read_csv` reads a data file into a *DataFrame*, a table with named columns, and
> `parse_dates=[...]` turns text such as `"2007-12-01"` into real dates. `zip(recessions.peak, recessions.trough)`
> hands over one (peak, trough) pair at a time, and `fig.add_vrect` shades the band between two dates.

In [10]:
recessions = pd.read_csv(DATA_URL + "recessions.csv", parse_dates=["peak", "trough"])


def shade_recessions(fig, start):
    """Shade every NBER recession that ends after the date `start` (Figures 2, 9, 14-17)."""
    for peak, trough in zip(recessions.peak, recessions.trough):
        if trough >= pd.Timestamp(start):
            fig.add_vrect(x0=peak, x1=trough, fillcolor=BLUE, opacity=0.12, line_width=0)
    return fig


recessions.tail(4)

,peak,trough
7,1990-07-01,1991-03-01
8,2001-03-01,2001-11-01
9,2007-12-01,2009-06-01
10,2020-02-01,2020-04-01


## Figure 1: The structure of the short-run model

**Goal.** See the whole short-run model at once: how the central bank's nominal interest rate ends up changing
inflation.

The textbook's Figure 12.1 is a chain of four circles. Each arrow is one building block:

$$i_t \;\xrightarrow{\text{MP curve}}\; R_t \;\xrightarrow{\text{IS curve}}\; \tilde{Y}_t \;\xrightarrow{\text{Phillips curve}}\; \Delta\pi_t$$

Here the chain is live. You choose the **inputs**: the nominal interest rate $i$ that the central bank sets, the
current (sticky) inflation rate $\pi$, and the demand shock $\bar{a}$. The circles show the **outcomes** that the
model computes: the real interest rate, short-run output and the change in inflation. The baseline is normal
times: inflation of 2 percent and a nominal rate of 7 percent, so that $R = 5\% = \bar{r}$ and output is at potential.

> **Python note.** `fig.add_shape(type="circle", ...)` draws a circle inside the box from (`x0`, `y0`) to (`x1`, `y1`),
> and `add_annotation(..., showarrow=False)` writes text at a point. `for k in range(4):` repeats the indented lines
> for k = 0, 1, 2, 3, so `names[k]` picks one name at a time. `scaleanchor="x"` gives both axes the same scale,
> so the circles stay round, and `visible=False` hides the axes.

In [11]:
def short_run_model(i=7.0, pi=2.0, a=0.0):
    """Figure 12.1: the nominal rate sets R (MP), R sets Y~ (IS), and Y~ sets the change in inflation (PC)."""
    R = real_rate(i, pi)
    Y = is_curve(R, a)
    dpi = phillips(Y)
    names = ["Nominal<br>interest rate, i", "Real interest<br>rate, R", "Short-run<br>output, Ỹ", "Change in<br>inflation, Δπ"]
    values = [f"{i:.1f}%", f"{R:.1f}%", f"{Y:+.1f}%", f"{dpi:+.2f}"]
    links = ["MP<br>curve", "IS<br>curve", "Phillips<br>curve"]
    fig = go.Figure()
    for k in range(4):
        fig.add_shape(type="circle", x0=3.4 * k - 1.15, x1=3.4 * k + 1.15, y0=-1.15, y1=1.15,
                      line=dict(color=TEAL, width=3), fillcolor="#e6f5f1")
        fig.add_annotation(x=3.4 * k, y=0.35, text=names[k], showarrow=False, font=dict(size=12))
        fig.add_annotation(x=3.4 * k, y=-0.45, text=f"<b>{values[k]}</b>", showarrow=False, font=dict(size=20, color=BLUE))
    for k in range(3):
        add_arrow(fig, 3.4 * k + 1.2, 0.0, 3.4 * k + 2.2, 0.0)
        fig.add_annotation(x=3.4 * k + 1.7, y=0.5, text=links[k], showarrow=False, font=dict(size=12))
    fig.add_annotation(x=5.1, y=-1.7, showarrow=False, font=dict(size=14),
                       text=f"Inflation next year: {pi:.1f}% + ({dpi:+.2f}) = {pi + dpi:.2f}%")
    fig.update_xaxes(visible=False, range=[-1.5, 11.7])
    fig.update_yaxes(visible=False, range=[-2.2, 1.8], scaleanchor="x")
    fig.update_layout(title=f"i = {i:.1f}%, π = {pi:.1f}%  →  R = {R:.1f}%  →  Ỹ = {Y:+.1f}%  →  Δπ = {dpi:+.2f}")
    return fig

> **Python note.** `interact` turns each argument of a function into a slider and calls the function again
> every time you move one. The `;` at the end only hides an extra line of text.

In [12]:
interact(short_run_model,
         i=FloatSlider(value=7.0, min=2.0, max=12.0, step=0.5, description="nominal interest rate i (%)", style=WIDE),
         pi=FloatSlider(value=2.0, min=0.0, max=8.0, step=0.5, description="sticky inflation π (%)", style=WIDE),
         a=FloatSlider(value=0.0, min=-4.0, max=4.0, step=0.5, description="demand shock ā", style=WIDE));

In [13]:
# @title Website version: Figure 1 with a precomputed slider for the nominal interest rate
slider_figure(short_run_model, "i", [5.0, 6.0, 7.0, 8.0, 9.0], start=7.0, label="nominal interest rate i (%)")

**What to notice.**

- Raise $i$ from 7 to 8 percent with inflation stuck at 2 percent: $R$ rises to 6 percent, output falls to
  $-2$ percent, and inflation starts to fall by 1 point a year. This is the textbook's summary of the model
  "in a nutshell": $\uparrow i \Rightarrow \uparrow R \Rightarrow \downarrow \tilde{Y} \Rightarrow \downarrow \Delta\pi$.
- What matters is $i - \pi$. Raise inflation and the nominal rate by the same amount: nothing real changes.
- A negative demand shock lowers output at any interest rate; the central bank can offset it by cutting $i$.

**Your turn.** Inflation is stuck at 4 percent and there is no demand shock. What nominal interest rate keeps the
economy at potential? Find it with the sliders, then check it with `real_rate` and `is_curve`.

<details><summary>Hint</summary>

Output is at potential when $R = \bar{r} = 5$ percent, so you need $i - 4 = 5$. Try `is_curve(real_rate(9.0, 4.0))`.

</details>

In [14]:
# Your turn: try it here

## Figure 2: The federal funds rate

**Goal.** See how much the Federal Reserve's main policy rate has moved since 1960.

The **federal funds rate** is the interest rate that banks charge each other for overnight loans. The Fed controls
it because it stands ready to lend and borrow at a stated rate: no bank pays more than the Fed charges, and no
bank lends for less than the Fed pays, or others would borrow cheaply and lend back at the higher rate (an
*arbitrage* opportunity).

In [15]:
fedfunds = pd.read_csv(DATA_URL + "fig12-2.csv", parse_dates=["date"])
fedfunds.tail()

,date,fedfunds
796,2026-05-01,3.63
797,2026-06-01,3.63
798,2026-07-01,3.63
799,2026-08-01,3.63
800,2026-09-01,3.75


Each row is a month. `fedfunds` is the effective federal funds rate in percent, the average of the daily rates of
that month. Let us find the highest and the lowest values.

> **Python note.** `.idxmax()` gives the row label of the largest value, and `.loc[row]` picks that row.
> `fedfunds.date.dt.year` is the year of each date, and `.between(2009, 2015)` is true for the years from 2009 to 2015.

In [16]:
peak = fedfunds.loc[fedfunds.fedfunds.idxmax()]
latest = fedfunds.iloc[-1]                                  # the last row
zero_years = fedfunds[fedfunds.date.dt.year.between(2009, 2015)]
print(f"Highest: {peak.fedfunds:.2f}% in {peak.date:%B %Y}")
print(f"2009-2015: between {zero_years.fedfunds.min():.2f}% and {zero_years.fedfunds.max():.2f}%")
print(f"Latest: {latest.fedfunds:.2f}% in {latest.date:%B %Y}")

Highest: 19.10% in June 1981
2009-2015: between 0.07% and 0.24%
Latest: 3.75% in September 2026


The chart can zoom in on three episodes of this chapter. Choose one from the menu.

> **Python note.** `px.line` draws a line chart straight from a DataFrame; `labels={...}` gives the axes readable
> names. A *dictionary* such as `PERIODS` maps each menu entry to its settings, and `PERIODS[period]` looks one up.
> A `Dropdown` is a menu: the function receives the text of the chosen entry.

In [17]:
PERIODS = {"1960–2026, all the data": (["1959-06-01", "2027-01-01"], [-0.8, 20]),
           "The Volcker years, 1977–1987": (["1977-01-01", "1987-12-31"], [-0.8, 20]),
           "Near-zero rates after 2008, 2006–2016": (["2006-01-01", "2016-12-31"], [-0.3, 8]),
           "Covid-19 and after, 2019–2026": (["2019-01-01", "2026-12-31"], [-0.3, 8])}


def fed_chart(period="1960–2026, all the data"):
    """Figure 12.2: the effective federal funds rate, with recessions shaded, over one period."""
    x_range, y_range = PERIODS[period]
    fig = px.line(fedfunds, x="date", y="fedfunds", labels={"date": "Year", "fedfunds": "Fed funds rate (percent)"},
                  title=f"The fed funds rate: {peak.fedfunds:.1f}% at its 1981 peak, near zero in 2009–15 and 2020–21")
    fig.update_traces(line_color=BLUE)
    shade_recessions(fig, "1960-01-01")
    fig.add_annotation(x=peak.date, y=peak.fedfunds, text=f"{peak.date:%B %Y}: {peak.fedfunds:.1f}%",
                       ax=90, ay=0, arrowhead=2)
    fig.add_annotation(x=latest.date, y=latest.fedfunds, text=f"{latest.date:%b %Y}: {latest.fedfunds:.2f}%",
                       ax=-40, ay=-50, arrowhead=2)
    fig.update_layout(xaxis_range=x_range, yaxis_range=y_range)
    return fig

In [18]:
interact(fed_chart, period=Dropdown(options=list(PERIODS), value="1960–2026, all the data", description="period", style=WIDE));

In [19]:
# @title Website version: Figure 2 with a menu for the period shown
fig = fed_chart()
buttons = [dict(label=name, method="relayout", args=[{"xaxis.range": x, "yaxis.range": y}])
           for name, (x, y) in PERIODS.items()]
fig.update_layout(updatemenus=[dict(type="dropdown", buttons=buttons, x=0.99, y=0.98, xanchor="right", yanchor="top")])
fig

**What to notice.**

- The rate has ranged from about 0.05 percent (2020–2021) to 19.1 percent (June 1981): the textbook's "nearly
  20 percent".
- Under Paul Volcker (Fed chair from 1979) the rate reached 17.6 percent in 1980 and above 19 percent in 1981, the
  tight money that ended the Great Inflation (Figures 10–12).
- The rate was close to zero in 2009–2015 and again in 2020–2021, then rose quickly in 2022–2023 to fight the
  return of inflation. Rate cuts followed from September 2024.
- The rate usually falls during and after recessions (the shaded bands): the Fed lowers it to stimulate the economy.

**Your turn.** How many months since December 2008 has the fed funds rate been below 0.25 percent? Count them with a
filter.

<details><summary>Hint</summary>

Keep the rows with `(fedfunds.date >= "2008-12-01") & (fedfunds.fedfunds < 0.25)` and count them with `len(...)`.

</details>

In [20]:
# Your turn: try it here

## Figure 3: The MP curve in the IS/MP diagram

**Goal.** Put the MP curve and the IS curve in one diagram and read off short-run output.

With sticky inflation, the central bank picks the real interest rate. The **MP curve** is simply a horizontal line at
that rate. The IS curve then tells us the output that goes with it: the economy is where the two curves cross.
This is the **IS/MP diagram**. When the central bank sets $R = \bar{r}$ and there is no demand shock, the curves cross
at $\tilde{Y} = 0$, point $A$: output is at potential.

The textbook uses this diagram for three figures: the starting point (Figure 12.3), a higher interest rate
(Figure 12.4), and the Volcker tightening (Figure 12.10). So we write **one** function, `ismp_experiment`, for all
three. Its arguments are the new values: the real interest rate `R_new` that the central bank sets, the demand
shock `a_new` and the slope `b`. With the defaults nothing changes. When a curve moves, the old one stays behind as
a dashed gray line.

> **Python note.** `if ... else` runs one block of lines or the other. `(a_new, b) != (A_BAR, B_BAR)` is true as soon
> as one of the two numbers differs from the baseline.

In [21]:
def ismp_experiment(R_new=R_BAR, a_new=A_BAR, b=B_BAR):
    """Figures 12.3, 12.4, 12.10: the economy moves from A (R = r_bar, Y~ = 0) to B, where IS' meets MP'."""
    Y_B = is_curve(R_new, a_new, b)
    fig = go.Figure()
    if (a_new, b) != (A_BAR, B_BAR):
        add_is(fig, "IS", GRAY, dash="dash")
        add_is(fig, "IS′", TEAL, a_new, b)
    else:
        add_is(fig, "IS", TEAL)
    if R_new != R_BAR:
        add_line(fig, MP_X, [R_BAR, R_BAR], "MP", GRAY, "dash")
        add_line(fig, MP_X, [R_new, R_new], "MP′", BLUE)
        add_arrow(fig, -7.3, R_BAR, -7.3, R_new)                 # the central bank moves R
    else:
        add_line(fig, MP_X, [R_BAR, R_BAR], "MP", BLUE)
    add_point(fig, 0.0, R_BAR, "A", -8, 0)
    if (Y_B, R_new) != (0.0, R_BAR):
        add_point(fig, Y_B, R_new, "B", -8, 0)
        add_arrow(fig, 0.0, 0.6, Y_B, 0.6)                       # the change in output
    set_axes(fig, X_OUTPUT, Y_RATE)
    fig.update_layout(title=f"MP: R = {R_new:.2f}%;  IS: ā = {a_new:.1f}, b̄ = {b:.2f}  →  Ỹ = {Y_B:+.1f}%")
    return fig

Start with the textbook's Figure 12.3: the central bank sets $R = \bar{r} = 5$ percent.

In [22]:
interact(ismp_experiment,
         R_new=FloatSlider(value=5.0, min=3.0, max=7.0, step=0.25, description="real interest rate R (%)", style=WIDE),
         a_new=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.5, description="demand shock ā", style=WIDE),
         b=FloatSlider(value=2.0, min=1.0, max=2.5, step=0.25, description="sensitivity of investment b̄", style=WIDE));

In [23]:
# @title Website version: Figure 3 with a precomputed slider for the demand shock
slider_figure(ismp_experiment, "a_new", [-2.0, -1.0, 0.0, 1.0, 2.0], start=0.0, label="demand shock ā")

**What to notice.**

- At the baseline the MP line and the IS curve cross at $A$: $R = \bar{r} = 5$ percent and $\tilde{Y} = 0$.
- The central bank chooses the height of the MP line, nothing else. The IS curve then decides how much output that
  interest rate brings.
- A demand shock moves the IS curve, and with $R$ unchanged output moves one for one with $\bar{a}$.

**Your turn.** Suppose $\bar{a} = 1$, a small boom. What real interest rate brings output back to potential? Find it
with the sliders, then check it with `rate_for_output`.

<details><summary>Hint</summary>

Solve $0 = 1 - 2\,(R - 5)$, or call `rate_for_output(0.0, a=1.0)`.

</details>

In [24]:
# Your turn: try it here

## Figure 4: Raising the interest rate in the IS/MP diagram

**Goal.** See how the central bank can cause a recession.

Inflation is slow to adjust, so a higher nominal interest rate means a higher real interest rate. The MP line moves
up to $R'$. The real interest rate is now above the marginal product of capital, so firms and households cut their
investment, and the economy moves **along** the IS curve from $A$ to $B$. The textbook's figure has no numbers; with
our baseline, $R' = 6$ percent gives the "$-2$ percent" of its later examples. It is the same function as in Figure 3.

In [25]:
interact(ismp_experiment,
         R_new=FloatSlider(value=6.0, min=3.0, max=7.0, step=0.25, description="new real interest rate R′ (%)", style=WIDE),
         a_new=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.5, description="demand shock ā′", style=WIDE),
         b=FloatSlider(value=2.0, min=1.0, max=2.5, step=0.25, description="sensitivity of investment b̄", style=WIDE));

In [26]:
# @title Website version: Figure 4 with a precomputed slider for the new real interest rate
slider_figure(ismp_experiment, "R_new", [5.0, 5.5, 6.0, 6.5, 7.0], start=6.0, label="new real interest rate R′ (%)")

**What to notice.**

- With $R' = 6$ percent, output falls from 0 to $-2$ percent: each percentage point on $R$ costs $\bar{b} = 2$
  points of output.
- The IS curve does not move: a change in $R$ is a movement **along** it. Only the MP line shifts.
- A flatter IS curve (a larger $\bar{b}$) makes the same rate increase more painful: with $\bar{b} = 2.5$ output falls
  to $-2.5$ percent.

**Your turn.** The Fed wants to cool a booming economy with $\bar{a} = 1.5$ and bring output exactly back to potential.
How high must $R'$ be? Set the sliders and check your answer with `is_curve`.

<details><summary>Hint</summary>

You need $1.5 - 2\,(R' - 5) = 0$. Check with `is_curve(5.75, a=1.5)`.

</details>

In [27]:
# Your turn: try it here

## Figure 5: Stabilizing the economy after a housing bubble

**Goal.** See how monetary policy can offset an aggregate demand shock.

When a housing bubble bursts, households lose wealth and confidence, and spend less: the demand shock $\bar{a}$ falls
to $\bar{a}' < 0$. The IS curve shifts left, and at an unchanged interest rate the economy moves from $A$ to $B$, a
recession (panel a; the textbook uses $-2$ percent as its example). The central bank can respond by cutting the
real interest rate to

$$R' = \bar{r} + \frac{\bar{a}'}{\bar{b}},$$

the rate at which the new IS curve gives $\tilde{Y} = 0$ (our `rate_for_output`). Cheaper borrowing raises investment
and makes up for the fall in $\bar{a}$: the economy moves to $C$, back at potential (panel b).

> **Python note.** `make_subplots(rows=1, cols=2, subplot_titles=(...))` makes a figure with two panels side by side.
> Our drawing functions take `row` and `col`, so `add_point(..., 1, 2)` draws in the panel of row 1, column 2.
> `for col in [1, 2]:` draws the parts that both panels share.

In [28]:
def housing_bubble(a_new=-2.0, b=B_BAR):
    """Figure 12.5: (a) a fall in demand moves the economy from A to B; (b) the Fed lowers R to R' and reaches C."""
    Y_B = is_curve(R_BAR, a_new, b)
    R_new = rate_for_output(0.0, a_new, b)          # the rate at which IS' gives Y~ = 0
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                        subplot_titles=("(a) The bubble bursts: A → B", "(b) The Fed lowers R: B → C"))
    for col in [1, 2]:
        add_is(fig, "IS", GRAY, A_BAR, b, "dash", 1, col)
        add_is(fig, "IS′", TEAL, a_new, b, "solid", 1, col)
        add_point(fig, 0.0, R_BAR, "A", -8, 0, 1, col)
        add_point(fig, Y_B, R_BAR, "B", -8, 0, 1, col)
        set_axes(fig, X_OUTPUT, Y_RATE, 1, col)
    add_line(fig, MP_X, [R_BAR, R_BAR], "MP", BLUE, "solid", 1, 1)
    add_arrow(fig, 0.0, 0.6, Y_B, 0.6, 1, 1)
    add_line(fig, MP_X, [R_BAR, R_BAR], "MP", GRAY, "dash", 1, 2)
    add_line(fig, MP_X, [R_new, R_new], "MP′", BLUE, "solid", 1, 2)
    add_arrow(fig, -7.3, R_BAR, -7.3, R_new, 1, 2)
    add_point(fig, 0.0, R_new, "C", -8, 0, 1, 2)
    add_arrow(fig, Y_B, 0.6, 0.0, 0.6, 1, 2)
    fig.update_layout(title=f"ā′ = {a_new:.1f}: B at Ỹ = {Y_B:+.1f}%;  R′ = {R_new:.2f}% brings C back to Ỹ = 0.0%")
    return fig

In [29]:
interact(housing_bubble,
         a_new=FloatSlider(value=-2.0, min=-3.0, max=0.0, step=0.25, description="demand shock after the bust ā′", style=WIDE),
         b=FloatSlider(value=2.0, min=1.0, max=2.5, step=0.25, description="sensitivity of investment b̄", style=WIDE));

In [30]:
# @title Website version: Figure 5 with a precomputed slider for the demand shock
slider_figure(housing_bubble, "a_new", [-3.0, -2.5, -2.0, -1.5, -1.0, -0.5], start=-2.0, label="demand shock after the bust ā′")

**What to notice.**

- The bust alone (panel a) takes output to $-2$ percent. Cutting $R$ from 5 to 4 percent (panel b) brings it back to
  potential: in the model, monetary policy can fully offset a demand shock.
- The larger the shock, the larger the cut. The flatter the IS curve (larger $\bar{b}$), the smaller the cut needed.
- In practice this is hard. It takes time to see how large the shock is, and Milton Friedman's "long and variable
  lags" mean that interest rates need 6–18 months to have their main effect. In 2008 the housing bust turned out to
  be much more than an ordinary demand shock (Chapter 14).

**Your turn.** Suppose the Fed waits and cuts $R$ only to 4.5 percent. Where does output end up after a bust of
$\bar{a}' = -2$? Compute it with `is_curve`.

<details><summary>Hint</summary>

The economy is on the new IS curve, so pass the new demand shock: `is_curve(4.5, a=-2.0)`.

</details>

In [31]:
# Your turn: try it here

## Figure 6: The yield curve for U.S. Treasuries

**Goal.** See how interest rates at different maturities fit together, and what an inverted yield curve looks like.

The Fed sets an overnight rate, yet a firm that builds a factory borrows for years. The two are linked by the
**term structure of interest rates**: a long-term interest rate is roughly the **average of the short-term rates**
that investors expect over its life. Otherwise, buying one 5-year bond and buying five 1-year bonds in a row would
pay different returns, and everyone would switch to the better one. The **yield curve** plots the interest rate
(the yield) of government bonds against their maturity.

In [32]:
curves = pd.read_csv(DATA_URL + "fig12-6.csv")
curves

,date,yield_1m,yield_3m,yield_6m,yield_12m,yield_24m,yield_36m,yield_60m,yield_84m,yield_120m,yield_240m,yield_360m
0,2022-01-03,0.05,0.08,0.22,0.40,0.78,1.04,1.37,1.55,1.63,2.05,2.01
1,2023-04-03,4.70,4.90,4.88,4.60,3.97,3.73,3.52,3.48,3.43,3.78,3.64
2,2026-10-15,4.17,4.28,4.35,4.49,4.77,4.85,4.91,5.00,5.10,5.48,5.43


Each row is one day, and each column the yield, in percent, of a Treasury security with the maturity in months that
the column name gives: `yield_1m` is the 1-month bill and `yield_360m` the 30-year bond. The first two days are the
textbook's curves; the third is the latest day in the data. To draw one line per day we need a **long** table, with
one row per day and maturity.

> **Python note.** `melt` turns columns into rows: each `yield_...` column becomes a value in the new column
> `column`, next to its yield. `.map(MATURITY)` replaces each column name by a readable label from the dictionary.

In [33]:
MATURITY = {"yield_1m": "1 mo", "yield_3m": "3 mo", "yield_6m": "6 mo", "yield_12m": "1 yr", "yield_24m": "2 yr",
            "yield_36m": "3 yr", "yield_60m": "5 yr", "yield_84m": "7 yr", "yield_120m": "10 yr",
            "yield_240m": "20 yr", "yield_360m": "30 yr"}
DAYS = {"2022-01-03": "January 3, 2022 (textbook)", "2023-04-03": "April 3, 2023 (textbook)",
        "2026-10-15": "October 15, 2026 (latest)"}
long = curves.melt(id_vars="date", var_name="column", value_name="yield")
long["maturity"] = long.column.map(MATURITY)
long["day"] = long.date.map(DAYS)
long.head()

,date,column,yield,maturity,day
0,2022-01-03,yield_1m,0.05,1 mo,"January 3, 2022 (textbook)"
1,2023-04-03,yield_1m,4.70,1 mo,"April 3, 2023 (textbook)"
2,2026-10-15,yield_1m,4.17,1 mo,"October 15, 2026 (latest)"
3,2022-01-03,yield_3m,0.08,3 mo,"January 3, 2022 (textbook)"
4,2023-04-03,yield_3m,4.90,3 mo,"April 3, 2023 (textbook)"


As in the textbook, the maturities are equally spaced on the horizontal axis, not drawn to scale.

In [34]:
fig = px.line(long, x="maturity", y="yield", color="day", markers=True, hover_name="day",
              color_discrete_map={DAYS["2022-01-03"]: BLUE, DAYS["2023-04-03"]: TEAL, DAYS["2026-10-15"]: GREEN},
              labels={"maturity": "Maturity", "yield": "Yield (percent)", "day": "Yield curve on"},
              title="The yield curve sloped up in January 2022 and was inverted in April 2023")
fig.update_traces(line_width=3, marker_size=9)
fig.update_layout(yaxis_range=[0, 6], legend=dict(x=0.01, y=0.62, xanchor="left", yanchor="top"))
fig.show()

How did the curve get from the first shape to the second? The file `yield_curves_monthly.csv` holds the curve on
the first trading day of every month of 2022, 2023 and 2026. Press play.

> **Python note.** `animation_frame="date"` makes one frame per date, with a play button and a slider below the
> chart; it also works on the website. `category_orders` keeps the maturities in their natural order.

In [35]:
monthly = pd.read_csv(DATA_URL + "yield_curves_monthly.csv")
fig = px.line(monthly, x="maturity", y="yield", animation_frame="date", markers=True,
              category_orders={"maturity": list(MATURITY.values())},
              labels={"maturity": "Maturity", "yield": "Yield (percent)", "date": "Date"},
              title="Press play: short rates overtake long rates as the Fed raises rates in 2022")
fig.update_traces(line_width=3, marker_size=9)
fig.update_layout(yaxis_range=[0, 6])
fig.show()

The file has no curves for 2024 and 2025, so the animation jumps from December 2023 to January 2026.

**What to notice.**

- In January 2022 the Fed's rate was near zero: short yields were close to 0 and rose steadily with maturity, the
  usual shape.
- During 2022 the Fed raised its rate fast (Figure 2). Short yields climbed above 4 percent, faster than long yields:
  by early 2023 the curve was **inverted**, with 5- and 10-year yields below the 1-month yield. Markets expected the
  Fed to lower rates over the next years.
- By 2026 the curve slopes up again, at a higher level than in 2022.

### Key numbers: the inversion

A common measure of the slope is the 10-year yield minus the 3-month yield. We compute it for every month.

> **Python note.** `pivot` does the opposite of `melt`: it makes one row per date and one column per maturity
> (here the maturity in months), so `wide[120] - wide[3]` subtracts two columns.

In [36]:
wide = monthly.pivot(index="date", columns="months", values="yield")
spread = wide[120] - wide[3]                         # 10-year minus 3-month yield
print(f"First inverted month: {spread[spread < 0].index[0]};  most inverted: {spread.idxmin()} ({spread.min():.2f} points)")
book = curves.set_index("date")
print("April 3, 2023: 5-year minus 1-month", round(book.yield_60m["2023-04-03"] - book.yield_1m["2023-04-03"], 2),
      "; 10-year minus 1-month", round(book.yield_120m["2023-04-03"] - book.yield_1m["2023-04-03"], 2))
spread.round(2).tail(4)

First inverted month: 2022-11-01;  most inverted: 2023-06-01 (-1.89 points)
April 3, 2023: 5-year minus 1-month -1.18 ; 10-year minus 1-month -1.27


date
2026-07-01    0.63
2026-08-03    0.79
2026-09-01    0.87
2026-10-01    1.07
dtype: float64

The curve first inverted in November 2022 and stayed inverted through 2023. On the textbook's day in April 2023 the
5- and 10-year yields were 1.18 and 1.27 points below the 1-month yield, as the textbook says.

**Your turn.** Suppose investors expect the 1-year interest rate to be 4, 4.5, 5, 5.5 and 6 percent in the next five
years. Use the term-structure idea to estimate today's 5-year rate. Is the yield curve sloping up or down?

<details><summary>Hint</summary>

The 5-year rate is about the average of the five expected 1-year rates: `np.mean([4.0, 4.5, 5.0, 5.5, 6.0])`. Compare it
with today's 1-year rate of 4 percent.

</details>

In [37]:
# Your turn: try it here

## Figure 7: The Phillips curve

**Goal.** Draw the Phillips curve and see how the state of the economy changes inflation.

Imagine you run a firm and inflation has been 5 percent a year. You expect it to stay there, so normally you raise
your prices by 5 percent. If demand for your products is weak, you raise them by less, to win back customers; if
demand is strong, by more. When all firms do this, inflation equals last year's inflation plus an adjustment for
demand:

$$\Delta\pi_t = \bar{\nu}\,\tilde{Y}_t + \bar{o}.$$

A booming economy ($\tilde{Y} > 0$) raises inflation; a slumping one lowers it. The slope $\bar{\nu}$ measures how
strongly prices react to demand. As with the IS diagram, one function draws three textbook figures: the Phillips
curve itself (12.7), an oil shock (12.8) and a recession (12.11). Its arguments are short-run output `Y_new`, the
price shock `o_new` and the slope `nu`.

> **Python note.** `for x, y, anchor, text in REGIONS:` takes the list of (x, y, anchor, text) groups one at a time and
> unpacks each into four names. `dict(X_OUTPUT, title=...)` copies the settings of `X_OUTPUT` and replaces the title.

In [38]:
REGIONS = [(-7.6, 3.6, "left", "Δπ > 0: inflation rises"), (7.6, -3.6, "right", "Δπ < 0: inflation falls")]
X_SLUMP_BOOM = dict(X_OUTPUT, title="←  Slumping economy          Short-run output, Ỹ (%)          Booming economy  →")


def pc_experiment(Y_new=0.0, o_new=O_BAR, nu=NU):
    """Figures 12.7, 12.8, 12.11: the Phillips curve, a price shock o' that shifts it, and the economy at Y~."""
    dpi = phillips(Y_new, nu, o_new)
    fig = go.Figure()
    if o_new != O_BAR:
        add_pc(fig, "PC", GRAY, nu, O_BAR, "dash")
        add_pc(fig, "PC′", GREEN, nu, o_new)
        add_arrow(fig, 2.0, phillips(2.0, nu), 2.0, phillips(2.0, nu, o_new))      # the shift of the curve
    else:
        add_pc(fig, "Phillips curve", GREEN, nu)
    add_point(fig, 0.0, phillips(0.0, nu, o_new), "A", -8, -4)
    if Y_new != 0.0:
        add_point(fig, Y_new, dpi, "B", -8, -4)
        add_arrow(fig, 0.0, -3.7, Y_new, -3.7)                                        # the change in output
    for x, y, anchor, text in REGIONS:
        fig.add_annotation(x=x, y=y, text=text, showarrow=False, xanchor=anchor, font=dict(size=13, color=GRAY))
    set_axes(fig, X_SLUMP_BOOM, Y_DPI)
    fig.update_layout(title=f"Ỹ = {Y_new:+.1f}%, ō = {o_new:+.1f}  →  Δπ = {nu:.2f} × ({Y_new:+.1f}) + ({o_new:+.1f}) = {dpi:+.2f}")
    return fig

In [39]:
interact(pc_experiment,
         Y_new=FloatSlider(value=2.0, min=-4.0, max=4.0, step=0.5, description="short-run output Ỹ (%)", style=WIDE),
         o_new=FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.5, description="price shock ō", style=WIDE),
         nu=FloatSlider(value=0.5, min=0.25, max=1.0, step=0.05, description="slope ν̄", style=WIDE));

In [40]:
# @title Website version: Figure 7 with a precomputed slider for short-run output
slider_figure(pc_experiment, "Y_new", [-4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0], start=2.0,
              label="short-run output Ỹ (%)")

**What to notice.**

- At potential ($\tilde{Y} = 0$, point $A$) inflation is steady: this year's inflation equals last year's.
- An economy 2 percent above potential raises inflation by $0.5 \times 2 = 1$ point a year, and it keeps rising as
  long as the boom lasts.
- A steeper curve (larger $\bar{\nu}$) means prices react more to demand. With a flat curve it takes a deep recession
  to lower inflation by one point.

**A brief history.** A. W. Phillips first related the **level** of (wage) inflation to economic activity. In the 1960s
many economists concluded that a country could keep output high for ever by accepting, say, 5 instead of 2 percent
inflation. Milton Friedman and Edmund Phelps argued that firms would soon build the higher inflation into their
prices: holding output above potential would make inflation **rise** year after year, which is the version of the
curve used here. The rising inflation of the 1970s proved them right.

**Your turn.** With the baseline slope, how far below potential must output be to lower inflation by 1.5 points in
one year? Find it with the sliders, then check it with `phillips`.

<details><summary>Hint</summary>

Solve $-1.5 = 0.5\,\tilde{Y}$. Then try `phillips(-3.0)`.

</details>

In [41]:
# Your turn: try it here

## Figure 8: An oil price increase

**Goal.** See how a price shock moves inflation even when output is at potential.

When oil prices jump, gasoline, airline tickets and heating cost more at once, and firms that use oil as an input pass
the higher cost on to their prices. In the model this is a positive **price shock** $\bar{o}$: the Phillips curve shifts
up by $\bar{o}$ for one period. Even at potential, inflation now rises by $\bar{o}$. Economists call such increases
**cost-push inflation**, and the $\bar{\nu}\tilde{Y}$ term **demand-pull inflation**. Price shocks can also come from
other inputs (steel), from large union wage settlements ($\bar{o} > 0$), or from a large inflow of workers that holds
wages down ($\bar{o} < 0$). The textbook's figure has no number; we use $\bar{o} = 1$ point.

In [42]:
interact(pc_experiment,
         Y_new=FloatSlider(value=0.0, min=-4.0, max=4.0, step=0.5, description="short-run output Ỹ (%)", style=WIDE),
         o_new=FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.5, description="price shock ō", style=WIDE),
         nu=FloatSlider(value=0.5, min=0.25, max=1.0, step=0.05, description="slope ν̄", style=WIDE));

In [43]:
# @title Website version: Figure 8 with a precomputed slider for the price shock
slider_figure(pc_experiment, "o_new", [-2.0, -1.0, 0.0, 1.0, 2.0], start=1.0, label="price shock ō")

**What to notice.**

- The whole curve moves up by $\bar{o}$: at $\tilde{Y} = 0$, inflation rises by 1 point.
- To keep inflation steady after the shock, output would have to fall to $\tilde{Y} = -\bar{o}/\bar{\nu} = -2$ percent:
  a price shock confronts the central bank with a hard choice.
- **Quantity theory or Phillips curve?** Chapter 8 found that faster growth of real GDP *lowers* inflation; here a boom
  *raises* it. The two are about different things: the quantity theory is a long-run model in which more output means
  more **supply** of goods; here higher short-run output means more **demand** than the economy's potential.

### Key numbers: demand-pull and cost-push

The change in inflation has two parts, $\bar{\nu}\tilde{Y}$ and $\bar{o}$. A loop computes them for a few cases.

In [44]:
for Y, o in [(0.0, 1.0), (2.0, 0.0), (2.0, 1.0), (-2.0, 1.0)]:
    print(f"Y~ = {Y:+.1f}%, o = {o:+.1f}:  demand-pull {phillips(Y, o=0.0):+.2f} + cost-push {o:+.2f}"
          f" = {phillips(Y, o=o):+.2f} points")

Y~ = +0.0%, o = +1.0:  demand-pull +0.00 + cost-push +1.00 = +1.00 points
Y~ = +2.0%, o = +0.0:  demand-pull +1.00 + cost-push +0.00 = +1.00 points
Y~ = +2.0%, o = +1.0:  demand-pull +1.00 + cost-push +1.00 = +2.00 points
Y~ = -2.0%, o = +1.0:  demand-pull -1.00 + cost-push +1.00 = +0.00 points


The last line is the hard case: a recession of 2 percent exactly offsets an oil shock of 1 point.

**Your turn.** Exercise 9 of the textbook: a large wave of new immigrants slows wage growth, which we model as
$\bar{o} = -0.5$. If the Fed keeps output at potential, what happens to inflation this year? And if it keeps inflation
steady instead, what output does that allow?

<details><summary>Hint</summary>

First `phillips(0.0, o=-0.5)`. For steady inflation solve $0 = 0.5\,\tilde{Y} - 0.5$.

</details>

In [45]:
# Your turn: try it here

## Figure 9: Inflation in the United States

**Goal.** Look at U.S. inflation since 1960 and at the role of monetary policy in it.

In [46]:
inflation = pd.read_csv(DATA_URL + "fig12-9.csv")
inflation.tail()

,year,inflation_cpi
61,2021,4.68124
62,2022,7.99082
63,2023,4.12696
64,2024,2.95211
65,2025,2.63430


`inflation_cpi` is the percent change in the average consumer price index of each year. One data note: the October 2025
CPI was never collected (the federal government was shut down), so the 2025 value averages the other 11 months.

We add the fed funds rate of Figure 2, averaged over each year, and the **real interest rate** from the Fisher equation,
$R = i - \pi$. This is the *ex post* real rate, computed with the inflation that actually happened. Investors decide with
the inflation they *expect* (the *ex ante* rate), but in this chapter we ignore the difference.

> **Python note.** `groupby(...)` puts the months of each year in one group, and `.mean()` averages each group.
> `.map(annual_ff)` then looks up the average for each year of the inflation table. `pd.to_datetime` turns the year
> into a date (1 January), so the chart can shade the recessions.

In [47]:
annual_ff = fedfunds.groupby(fedfunds.date.dt.year).fedfunds.mean()
inflation["fed_funds"] = inflation.year.map(annual_ff)
inflation["real_rate"] = real_rate(inflation.fed_funds, inflation.inflation_cpi)
inflation["date"] = pd.to_datetime(inflation.year.astype(str))
top = inflation.loc[inflation.inflation_cpi.idxmax()]
inflation[inflation.year.between(1978, 1984)].round(2)

,year,inflation_cpi,fed_funds,real_rate,date
18,1978,7.63,7.93,0.30,1978-01-01
19,1979,11.25,11.19,-0.06,1979-01-01
20,1980,13.50,13.36,-0.15,1980-01-01
21,1981,10.38,16.38,6.00,1981-01-01
22,1982,6.16,12.26,6.10,1982-01-01
23,1983,3.16,9.09,5.93,1983-01-01
24,1984,4.37,10.22,5.86,1984-01-01


The chart shows inflation as a filled area. Click the legend to add the fed funds rate and the real interest rate.

In [48]:
fig = px.area(inflation, x="date", y="inflation_cpi", labels={"date": "Year", "inflation_cpi": "Percent"},
              title=f"Inflation peaked at {top.inflation_cpi:.1f}% in {top.year:.0f}; recessions usually bring it down")
fig.update_traces(name="Inflation (CPI)", showlegend=True, line_color=TEAL)
fig.add_trace(go.Scatter(x=inflation.date, y=inflation.fed_funds, name="Fed funds rate, i", visible="legendonly",
                         line=dict(color=BLUE, width=2)))
fig.add_trace(go.Scatter(x=inflation.date, y=inflation.real_rate, name="Real interest rate, R = i − π",
                         visible="legendonly", line=dict(color=GREEN, width=2, dash="dot")))
fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)
shade_recessions(fig, "1960-01-01")
fig.update_layout(yaxis_range=[-8, 18], legend=dict(x=0.99, y=0.98, xanchor="right", yanchor="top"))
fig.show()

**What to notice.**

- Inflation was low in the early 1960s (1.1 to 1.5 percent; the textbook's "about 2 percent" is generous), rose to
  11.0 percent in 1974 and peaked at 13.5 percent in 1980, then fell to 3.2 percent by 1983: the Volcker disinflation.
- Turn on the real interest rate: it was close to zero in 1979–1980 and jumped to about 6 percent in 1981–1984. That
  is the tight monetary policy of Figures 10–12.
- Inflation usually falls after recessions (the shaded bands), as the Phillips curve predicts.
- In 2021–2022 inflation returned (8.0 percent in 2022), while the real interest rate was deeply negative: monetary
  policy was very loose (Figure 15).

**Your turn.** In which year since 1960 was the real interest rate lowest, and in which was it highest? Use `idxmin`
and `idxmax` on the column `real_rate`.

<details><summary>Hint</summary>

`inflation.loc[inflation.real_rate.idxmin()]` gives the whole row of the lowest value; do the same with `idxmax`.

</details>

In [49]:
# Your turn: try it here

## Figure 10: Tightening monetary policy

**Goal.** Follow the first step of the Volcker disinflation in the IS/MP diagram.

By 1979 inflation exceeded 10 percent. Paul Volcker's Fed slowed money growth, which means raising the nominal
interest rate. Because inflation is sticky, the real rate rises too: $R' = i' - \pi$. With inflation at 10 percent, a
nominal rate of 15 percent keeps $R = 5\% = \bar{r}$; raising it to 16 percent lifts the real rate to 6 percent, and output
falls to the textbook's $-2$ percent. The function reuses `ismp_experiment` and only adds the Fisher step.

In [50]:
def tightening(i_new=16.0, pi=10.0):
    """Figure 12.10: with sticky inflation pi, a higher nominal rate i' raises R' = i' - pi and lowers output."""
    R_new = real_rate(i_new, pi)
    fig = ismp_experiment(R_new=R_new)
    fig.update_layout(title=f"i′ = {i_new:.1f}% and sticky π = {pi:.1f}%  →  R′ = {R_new:.1f}%  →  Ỹ = {is_curve(R_new):+.1f}%")
    return fig

In [51]:
interact(tightening,
         i_new=FloatSlider(value=16.0, min=13.0, max=18.0, step=0.5, description="new nominal interest rate i′ (%)", style=WIDE),
         pi=FloatSlider(value=10.0, min=9.0, max=11.0, step=0.5, description="sticky inflation π (%)", style=WIDE));

In [52]:
# @title Website version: Figure 10 with a precomputed slider for the nominal interest rate
slider_figure(tightening, "i_new", [15.0, 15.5, 16.0, 16.5, 17.0, 17.5, 18.0], start=16.0, label="new nominal interest rate i′ (%)")

**What to notice.**

- Only the real rate matters for output: $i' = 16$ with $\pi = 10$ has the same effect as $i' = 7$ with $\pi = 1$.
- If inflation were not sticky (if the classical dichotomy held even in the short run), a higher nominal rate could
  simply come with lower inflation and leave output alone. Stickiness is what gives monetary policy real effects.
- In 1981 the effective fed funds rate averaged 16.4 percent, and the real rate was about 6 percent (Figure 9): the
  deepest recession in decades followed.

**Your turn.** Inflation stays at 10 percent. What nominal interest rate would push output to $-3$ percent? Find it with
the slider, then check it with `rate_for_output`.

<details><summary>Hint</summary>

`rate_for_output(-3.0)` gives the real rate; add the 10 percent inflation to get the nominal rate.

</details>

In [53]:
# Your turn: try it here

## Figure 11: A recession and falling inflation

**Goal.** Follow the second step: the recession lowers inflation through the Phillips curve.

The Fed's recession takes output to $\tilde{Y}' = -2$ percent. Firms see demand fall, so they raise prices less
aggressively, and the economy jumps from $A$ to $B$ on the Phillips curve, where $\Delta\pi' < 0$. It is the
`pc_experiment` function again, now with output below potential.

In [54]:
interact(pc_experiment,
         Y_new=FloatSlider(value=-2.0, min=-4.0, max=4.0, step=0.5, description="output in the recession Ỹ′ (%)", style=WIDE),
         o_new=FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.5, description="price shock ō", style=WIDE),
         nu=FloatSlider(value=0.5, min=0.25, max=1.0, step=0.05, description="slope ν̄", style=WIDE));

In [55]:
# @title Website version: Figure 11 with a precomputed slider for output in the recession
slider_figure(pc_experiment, "Y_new", [-4.0, -3.0, -2.0, -1.0, 0.0], start=-2.0, label="output in the recession Ỹ′ (%)")

**What to notice.**

- With $\bar{\nu} = 0.5$, a recession of 2 percent lowers inflation by 1 point: from 10 to 9 percent in the first year.
- The textbook's example, "instead of raising prices by 10 percent, they may raise them by only 8 percent", is a fall
  of 2 points; it corresponds to $\bar{\nu} = 1$. Move the slope slider to 1 to see it.
- The cost of lower inflation is lost output; the steeper the curve, the cheaper the disinflation.

**Your turn.** Exercise 11 asks what a steeper Phillips curve means for the Volcker disinflation. Compute the fall in
inflation in one year at $\tilde{Y} = -2$ for $\bar{\nu} = 0.25$, 0.5 and 1 with a loop.

<details><summary>Hint</summary>

`for nu in [0.25, 0.5, 1.0]:` and inside it `print(nu, phillips(-2.0, nu))`.

</details>

In [56]:
# Your turn: try it here

## Figure 12: The disinflation over time

**Goal.** Run the Volcker policy year by year and watch inflation come down.

The Fed raises the real interest rate in year 0 and keeps it high until year $t^*$, when inflation has fallen enough;
then it returns $R$ to $\bar{r}$. Each year the model takes three steps: the MP curve sets $R_t$, the IS curve gives
$\tilde{Y}_t$, and the Phillips curve gives $\pi_t = \pi_{t-1} + \bar{\nu}\tilde{Y}_t$. That is exactly our `simulate`
function. Inflation starts at the textbook's 10 percent. Inflation in year 0 already falls, because
$\pi_0 = (P_1 - P_0)/P_0$: firms set next year's prices when they see this year's recession.

> **Python note.** `np.arange(-2, 10)` is the years $-2, -1, \ldots, 9$. `np.where(condition, a, b)` gives `a` where the
> condition is true and `b` elsewhere: here a high rate from year 0 to $t^* - 1$ and $\bar{r}$ otherwise. `&` means "and".
> `line_shape="hv"` draws steps: a flat line, then a jump. `IntSlider` is a slider for whole numbers.

In [57]:
def disinflation(Y_rec=-2.0, t_star=5, nu=NU):
    """Figure 12.12: the Fed holds R high from year 0 to t*; output is Y_rec and inflation falls from 10%."""
    years = np.arange(-2, 10)
    R_path = np.where((years >= 0) & (years < t_star), rate_for_output(Y_rec), R_BAR)
    Y_path, pi_path = simulate(R_path, pi_start=10.0, nu=nu)
    fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.09,
                        subplot_titles=("(a) The Fed raises R ...", "(b) causing a recession ...", "(c) which lowers inflation"))
    paths = [R_path, Y_path, pi_path]
    colors, shapes = [BLUE, TEAL, GREEN], ["hv", "hv", "linear"]
    axes = [dict(title="Real interest rate, R (%)", range=[2.5, 7.5]), dict(title="Short-run output, Ỹ (%)", range=[-4.5, 1.5]),
            dict(title="Inflation, π (%)", range=[0, 11.5])]
    for k in range(3):
        fig.add_trace(go.Scatter(x=years, y=paths[k], mode="lines+markers", showlegend=False, line_shape=shapes[k],
                                 line=dict(color=colors[k], width=4), marker=dict(color="black", size=9)), row=1, col=k + 1)
        set_axes(fig, dict(title="Year", tickvals=[0, t_star], ticktext=["0", "t*"], range=[-2.5, 9.5]), axes[k], 1, k + 1)
    fig.update_layout(title=f"π: 10.0% → {pi_path[-1]:.1f}% after {t_star} years at Ỹ = {Y_rec:+.1f}%;"
                            f"  output lost: {-Y_path.sum():.1f} %-years")
    return fig

This time we use `interact_manual`. It adds a button, so nothing happens until you press it. Choose the depth of the
recession, the years of tight policy and the slope, then press **Run the Volcker policy**.

In [58]:
run_volcker = interact_manual.options(manual_name="Run the Volcker policy")
run_volcker(disinflation,
            Y_rec=FloatSlider(value=-2.0, min=-4.0, max=-0.5, step=0.5, description="output in the recession (%)", style=WIDE),
            t_star=IntSlider(value=5, min=1, max=9, step=1, description="years of tight policy t*", style=WIDE),
            nu=FloatSlider(value=0.5, min=0.25, max=1.0, step=0.05, description="slope ν̄", style=WIDE));

In [59]:
# @title Website version: Figure 12 with a precomputed slider for the years of tight policy
slider_figure(disinflation, "t_star", [1, 2, 3, 4, 5, 6, 7, 8], start=5, label="years of tight policy t*")

**What to notice.**

- Five years at $\tilde{Y} = -2$ percent bring inflation from 10 to 5 percent, one point a year. When the Fed returns
  $R$ to $\bar{r}$, output goes back to potential and inflation stays at its new, lower level.
- The price is 10 "percent-years" of lost output: 2 percent of GDP for five years, or 2 percent-years for each point of
  inflation removed.
- A deeper recession gets there faster, at about the same total cost: with $\tilde{Y} = -4$ it takes 2.5 years. Only a
  steeper Phillips curve makes the disinflation cheaper.
- In reality, inflation fell from 13.5 percent in 1980 to 3.2 percent in 1983 (Figure 9), faster than in this
  example, and the recession of 1981–1982 was the deepest in decades.

**Your turn.** The Fed wants inflation to fall from 10 to 4 percent in exactly four years. How deep must the recession
be with $\bar{\nu} = 0.5$? Find it with the sliders, then check it with `simulate`.

<details><summary>Hint</summary>

Inflation must fall by 1.5 points a year, so $\bar{\nu}\tilde{Y} = -1.5$. The real rate is `rate_for_output(-3.0)`; pass it
four times, then `R_BAR`: `simulate([6.5, 6.5, 6.5, 6.5, 5.0], pi_start=10.0)`.

</details>

In [60]:
# Your turn: try it here

## Figure 13: Mistaking a slowdown in potential for a recession

**Goal.** See how a slowdown in potential output, mistaken for a recession, can feed inflation.

From the early 1970s, U.S. productivity growth slowed down, so **potential** output grew more slowly. At the time,
policymakers thought potential output was still growing at its old rate. They saw actual output fall behind that
believed path, read it as a recession ($\tilde{Y}_{\text{perceived}} < 0$), and lowered interest rates. In truth output
was then **above** its true potential ($\tilde{Y}_{\text{true}} > 0$), and through the Phillips curve inflation rose.

The textbook's figure is stylized, so we describe it with three numbers: potential growth before 1973 (`g_old`), after
1973 (`g_new`), and how far the stimulus eventually pushes output above true potential (`gap_max`, reached gradually).
Output is an index with 1973 = 100. This small model is used only here, so we define it in this section.

> **Python note.** `np.exp` is the exponential function: an index growing at g percent a year is
> $100\,e^{g\,(t - 1973)/100}$. `np.minimum(x, 0)` keeps the negative values of `x` and sets the others to 0, and
> `np.maximum(x, 0)` does the opposite. `years == 1980` is true only in the year 1980.

In [61]:
def output_paths(g_old=3.5, g_new=2.5, gap_max=4.0):
    """Figure 12.13: believed potential, true potential and actual output (1973 = 100)."""
    years = np.arange(1965, 1986)
    before, after = np.minimum(years - 1973, 0), np.maximum(years - 1973, 0)
    believed = 100 * np.exp(g_old * (years - 1973) / 100)
    true = 100 * np.exp((g_old * before + g_new * after) / 100)
    gap_true = gap_max * (1 - np.exp(-after / 4))      # the stimulus pushes output above true potential
    actual = true * (1 + gap_true / 100)
    return years, believed, true, actual

In [62]:
def slowdown(g_old=3.5, g_new=2.5, gap_max=4.0):
    """Figure 12.13 with readouts: the perceived and the true output gap, and the rise in inflation."""
    years, believed, true, actual = output_paths(g_old, g_new, gap_max)
    perceived_gap, true_gap = 100 * (actual / believed - 1), 100 * (actual / true - 1)
    dpi = phillips(true_gap[(years > 1973) & (years <= 1980)]).sum()   # Phillips curve, 1974-1980
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=years, y=believed, name="Believed path of potential output", line=dict(color=GREEN, width=4, dash="dash")))
    fig.add_trace(go.Scatter(x=years, y=actual, name="Actual output", line=dict(color=TEAL, width=4)))
    fig.add_trace(go.Scatter(x=years, y=true, name="True potential output", line=dict(color=BLUE, width=4, dash="dash")))
    gaps = [(actual[-1], believed[-1], "Ỹ perceived < 0"), (true[-1], actual[-1], "Ỹ true > 0")]
    for low, high, text in gaps:
        fig.add_trace(go.Scatter(x=[1985.4, 1985.4], y=[low, high], mode="lines", showlegend=False, line=dict(color="black", width=2)))
        fig.add_annotation(x=1985.6, y=(low + high) / 2, text=text, showarrow=False, xanchor="left", font=dict(size=13))
    k = 1980 - 1965                                    # the position of 1980 in the arrays
    fig.add_vline(x=1973, line_dash="dot", line_color=GRAY)
    fig.update_layout(title=f"1980: perceived Ỹ = {perceived_gap[k]:+.1f}%, true Ỹ = {true_gap[k]:+.1f}%;"
                            f" inflation +{dpi:.1f} points since 1973",
                      xaxis=dict(title="Year", range=[1964, 1990], tickvals=[1965, 1970, 1973, 1980, 1985]),
                      yaxis=dict(title="Output (index, 1973 = 100)", range=[70, 150]),
                      legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top"))
    return fig

In [63]:
interact(slowdown,
         g_old=FloatSlider(value=3.5, min=2.0, max=5.0, step=0.25, description="potential growth before 1973 (%)", style=WIDE),
         g_new=FloatSlider(value=2.5, min=1.0, max=4.0, step=0.25, description="true potential growth after 1973 (%)", style=WIDE),
         gap_max=FloatSlider(value=4.0, min=0.0, max=6.0, step=0.5, description="stimulus: long-run true gap (%)", style=WIDE));

In [64]:
# @title Website version: Figure 13 with a precomputed slider for true potential growth
slider_figure(slowdown, "g_new", [1.5, 2.0, 2.5, 3.0, 3.5], start=2.5, label="true potential growth after 1973 (%)")

**What to notice.**

- By 1980 actual output is 3.7 percent *below* the believed path, so policymakers see a recession and keep stimulating.
  Yet it is 3.3 percent *above* true potential.
- The true gap is positive every year after 1973, so the Phillips curve raises inflation every year: about 8 points by
  1980 with $\bar{\nu} = 0.5$. Actual U.S. inflation rose from 6.3 percent in 1973 to 13.5 percent in 1980, also helped by
  the oil shocks of 1974 and 1979.
- Set the growth rate after 1973 equal to the one before: there is no slowdown, the two dashed paths coincide, and the
  stimulus is plainly a boom.
- Policymakers learn of their mistake through rising inflation, but oil shocks also raise inflation, which made the
  lesson slow to learn.

### The short-run model in a nutshell

The textbook sums up how monetary policy works through the three building blocks:

| Building block | Step |
|---|---|
| MP curve | $\uparrow i_t \Rightarrow \uparrow R_t$ |
| IS curve | $\uparrow R_t \Rightarrow \downarrow \tilde{Y}_t$ |
| Phillips curve | $\downarrow \tilde{Y}_t \Rightarrow \downarrow \Delta\pi_t$ |

Here is the chain in numbers, with inflation stuck at 10 percent, for three nominal interest rates.

In [65]:
rows = []
for i in [15.0, 16.0, 17.0]:
    R = real_rate(i, 10.0)
    Y = is_curve(R)
    rows.append([i, R, Y, phillips(Y)])
pd.DataFrame(rows, columns=["nominal rate i (%)", "real rate R (%)", "output Y~ (%)", "change in inflation (points)"])

,nominal rate i (%),real rate R (%),output Y~ (%),change in inflation (points)
0,15.0,5.0,0.0,0.0
1,16.0,6.0,-2.0,-1.0
2,17.0,7.0,-4.0,-2.0


**Your turn.** In the slowdown model, how large must the stimulus (`gap_max`) be for inflation to rise by only 2 points
from 1973 to 1980? Try a few values with the slider.

<details><summary>Hint</summary>

The rise in inflation is proportional to `gap_max`: with 4.0 it is about 8 points. Check your guess with the slider's
readout.

</details>

In [66]:
# Your turn: try it here

## Figure 14: Real GDP in the United States

**Goal.** Measure how deep the Covid-19 recession was and how fast GDP came back.

In [67]:
gdp = pd.read_csv(DATA_URL + "fig12-14.csv", parse_dates=["date"])
gdp.head()

,date,real_gdp
0,2015-01-01,18.6666
1,2015-04-01,18.7822
2,2015-07-01,18.8574
3,2015-10-01,18.8922
4,2016-01-01,19.0017


Each row is a quarter. `real_gdp` is real GDP in trillions of chained 2017 dollars, at an annual rate. The textbook used
2012 dollars, so its numbers are 8–10 percent lower; the shape is the same. To judge whether GDP was "back on trend", we
fit a straight line to the logarithm of GDP before Covid (2015Q1–2019Q4), which means a constant growth rate, and extend it.

> **Python note.** `np.polyfit(x, y, 1)` fits a straight line $y = \text{slope}\cdot x + \text{intercept}$ by least squares.
> `np.log` and `np.exp` move between levels and logs; `np.exp(4 * slope)` turns quarterly log growth into annual growth.

In [68]:
pre = gdp[gdp.date <= "2019-10-01"]                     # 2015Q1 to 2019Q4
t = np.arange(len(gdp))                                 # quarters since 2015Q1
slope, intercept = np.polyfit(t[:len(pre)], np.log(pre.real_gdp), 1)
gdp["trend"] = np.exp(intercept + slope * t)
gdp["gap"] = 100 * (gdp.real_gdp / gdp.trend - 1)       # percent above (+) or below (-) the trend
fall = 100 * (gdp.real_gdp[gdp.date == "2020-04-01"].iloc[0] / gdp.real_gdp[gdp.date == "2019-10-01"].iloc[0] - 1)
print(f"Pre-Covid trend growth: {100 * (np.exp(4 * slope) - 1):.2f}% a year;  2019Q4 to 2020Q2: {fall:.2f}%")
gdp[gdp.date.dt.year.between(2020, 2022)].round(2)

Pre-Covid trend growth: 2.51% a year;  2019Q4 to 2020Q2: -9.09%


,date,real_gdp,trend,gap
20,2020-01-01,20.71,21.00,-1.39
21,2020-04-01,19.08,21.13,-9.72
22,2020-07-01,20.56,21.26,-3.31
23,2020-10-01,20.79,21.40,-2.82
24,2021-01-01,21.11,21.53,-1.96
25,2021-04-01,21.47,21.66,-0.87
26,2021-07-01,21.64,21.80,-0.71
27,2021-10-01,22.00,21.93,0.32
28,2022-01-01,21.95,22.07,-0.56
29,2022-04-01,21.98,22.21,-1.03


In [69]:
chart = gdp.rename(columns={"real_gdp": "Real GDP", "trend": "Pre-Covid trend (2015–2019)"})
fig = px.line(chart, x="date", y=["Real GDP", "Pre-Covid trend (2015–2019)"], markers=True,
              color_discrete_sequence=[BLUE, GRAY],
              labels={"date": "Year", "value": "Trillions of chained 2017 dollars", "variable": "Series"},
              title=f"Real GDP fell {-fall:.1f}% in the Covid-19 recession and was back on trend by late 2021")
fig.update_traces(line_dash="dash", marker_size=4, selector=dict(name="Pre-Covid trend (2015–2019)"))
shade_recessions(fig, "2015-01-01")
fig.update_layout(yaxis_range=[18, 25.5], legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top", title=None))
fig.show()

**What to notice.**

- From 2019Q4 to 2020Q2 real GDP fell by 9.1 percent, the textbook's "more than 10 percent" in round numbers (9.6
  percent in the data of 2023).
- The rebound was just as remarkable: from 2021Q2 on, GDP has stayed within about 1 percent of its pre-Covid trend. It
  was 0.3 percent above the trend in 2021Q4, but it never rose sharply above it.
- Before Covid, real GDP grew by about 2.5 percent a year.

**Your turn.** In which quarter was GDP furthest below its trend, and by how much? Use the column `gap`.

<details><summary>Hint</summary>

`gdp.loc[gdp.gap.idxmin()]` gives the whole row.

</details>

In [70]:
# Your turn: try it here

## Figure 15: Consumer price index inflation in the United States

**Goal.** See the surprising path of inflation during and after the pandemic.

In [71]:
cpi = pd.read_csv(DATA_URL + "fig12-15.csv", parse_dates=["date"])
cpi.head()

,date,inflation_all,inflation_core
0,2020-01-01,2.599770,2.28174
1,2020-02-01,2.341320,2.36766
2,2020-03-01,1.494040,2.09774
3,2020-04-01,0.313047,1.43093
4,2020-05-01,0.198201,1.21846


Each row is a month. `inflation_all` is the percent change of the consumer price index over the previous 12 months, and
`inflation_core` the same for **core** prices, which leave out food and energy, the most volatile prices. Are any values
missing?

> **Python note.** `.isna()` is true where a value is missing, and `.any(axis=1)` is true for rows with at least one
> missing value.

In [72]:
cpi[cpi.isna().any(axis=1)]

,date,inflation_all,inflation_core
69,2025-10-01,NaN,NaN


October 2025 is empty: the federal government was shut down that month and the Bureau of Labor Statistics did not collect
prices. We keep the gap rather than invent a number; the chart simply breaks the lines there.

In [73]:
chart = cpi.rename(columns={"inflation_all": "All items", "inflation_core": "Core (less food and energy)"})
fig = px.line(chart, x="date", y=["All items", "Core (less food and energy)"],
              labels={"date": "Year", "value": "Percent change over 12 months", "variable": "CPI"},
              title="CPI inflation fell to 0.2% in May 2020, then rose to 9.0% in June 2022")
fig.update_traces(line_width=3)
fig.add_hline(y=2, line_dash="dot", line_color=GRAY, annotation_text="2 percent", annotation_position="bottom right")
shade_recessions(fig, "2020-01-01")
fig.update_layout(yaxis=dict(range=[0, 10], tickvals=[2, 4, 6, 8, 10]),
                  legend=dict(x=0.99, y=0.98, xanchor="right", yanchor="top", title=None))
fig.show()

**What to notice.**

- At the start of the pandemic, inflation fell, as the Phillips curve predicts for a weak economy: from 2.6 percent in
  January 2020 to 0.2 percent in May 2020.
- Within a year it rose to levels not seen in four decades: 9.0 percent in June 2022 (the last higher reading was 9.6
  percent in November 1981). Core inflation peaked later, at 6.6 percent in September 2022.
- Inflation then came down, without a recession, to between 2.3 and 3.0 percent from mid-2024 through 2025. In 2026 headline inflation rose
  back to 3–4 percent while core inflation stayed near 2.5 percent, a sign of a new price shock ($\bar{o} > 0$).

**Your turn.** In which month did all-items inflation first fall below 3 percent after its 2022 peak?

<details><summary>Hint</summary>

Keep the rows after the peak with `cpi.date > "2022-06-01"` and `cpi.inflation_all < 3`, then take the first `date`.

</details>

In [74]:
# Your turn: try it here

## Figure 16: U.S. federal government spending as a share of GDP

**Goal.** See how strongly government spending reacted to the pandemic.

In [75]:
outlays = pd.read_csv(DATA_URL + "fig12-16.csv", parse_dates=["date"])
outlays.tail(8)

,date,outlays_gdp_pct
11,2018-01-01,19.8919
12,2019-01-01,20.6451
13,2020-01-01,30.4876
14,2021-01-01,28.7482
15,2022-01-01,24.0706
16,2023-01-01,22.0355
17,2024-01-01,22.9888
18,2025-01-01,22.7914


`outlays_gdp_pct` is federal net outlays in percent of GDP. The data are by **fiscal year** (October to September), and each
fiscal year is dated January 1, so FY2020 appears as `2020-01-01`.

In [76]:
fig = px.line(outlays, x="date", y="outlays_gdp_pct", markers=True,
              labels={"date": "Fiscal year", "outlays_gdp_pct": "Percent of GDP"},
              title="Federal spending jumped from 20.7% of GDP in FY2019 to 30.5% in FY2020")
fig.update_traces(line_color=BLUE, line_width=3, marker_size=9)
shade_recessions(fig, "2007-01-01")
fig.update_layout(yaxis_range=[16, 32])
fig.show()

**What to notice.**

- Spending rose from 20.7 percent of GDP in FY2019 to 30.5 percent in FY2020 and 28.8 percent in FY2021, then fell back to
  about 23 percent. The Great Recession had raised it too, to 24.3 percent in FY2009.
- Much of the extra spending was transfers to households and firms: loans, generous unemployment benefits and three
  rounds of direct payments (Figure 17).
- **Data check.** The textbook says spending rose "from 22 percent in 2019". Neither these data (20.65) nor the textbook's
  own figure (20.8) show that: it is a slip in the text. Its "nearly 32 percent in 2020 and 30 percent in 2021" match the
  data of 2023 (31.1 and 29.3); since then a revision of GDP has lowered these shares by about half a point.

**Your turn.** By how many percentage points of GDP did spending rise from FY2019 to FY2020, and from FY2008 to FY2009?
Which shock raised spending more?

<details><summary>Hint</summary>

`outlays.set_index("date").outlays_gdp_pct` lets you pick years with `.loc["2020-01-01"]`; subtract the two years.

</details>

In [77]:
# Your turn: try it here

## Figure 17: Income and consumption

**Goal.** See how government payments kept household income up during the pandemic.

In [78]:
income = pd.read_csv(DATA_URL + "fig12-17.csv", parse_dates=["date"])
income.head()

,date,real_disp_income,real_consumption
0,2017-01-01,14.3737,13.1363
1,2017-02-01,14.4168,13.1434
2,2017-03-01,14.4763,13.2017
3,2017-04-01,14.5093,13.2061
4,2017-05-01,14.6107,13.2145


Each row is a month. `real_disp_income` is real **disposable** personal income, income after taxes and government
transfers, and `real_consumption` real personal consumption, both in trillions of chained 2017 dollars at annual rates.
The U.S. government sent three payments to each person: 1,200 dollars in April 2020, 600 dollars in January 2021 and
1,400 dollars in March 2021. Let us find the three largest monthly jumps in income.

> **Python note.** `.diff()` gives the change from the previous row, and `.nlargest(3)` keeps the three largest values.

In [79]:
jumps = income.set_index("date").real_disp_income.diff().nlargest(3)
jumps.round(2)

date
2021-03-01    3.81
2020-04-01    2.31
2021-01-01    1.81
Name: real_disp_income, dtype: float64

The three largest jumps are exactly the three payment months. We mark them on the chart.

In [80]:
chart = income.rename(columns={"real_disp_income": "Disposable income", "real_consumption": "Consumption"})
fig = px.line(chart, x="date", y=["Disposable income", "Consumption"], color_discrete_sequence=[BLUE, TEAL],
              labels={"date": "Year", "value": "Trillions of chained 2017 dollars", "variable": "Series"},
              title="Government payments pushed disposable income above its pre-pandemic level")
fig.update_traces(line_width=3)
paid = income.set_index("date").loc[pd.to_datetime(["2020-04-01", "2021-01-01", "2021-03-01"])]
fig.add_trace(go.Scatter(x=paid.index, y=paid.real_disp_income, name="Payments (dollars per person)",
                         mode="markers+text", text=["1,200", "600", "1,400"], textposition=["top center", "top left", "top center"],
                         marker=dict(color="black", size=10)))
shade_recessions(fig, "2017-01-01")
fig.update_layout(yaxis_range=[11, 22], legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top", title=None))
fig.show()

**What to notice.**

- Consumption collapsed in April 2020, by 16.8 percent from February, and was back above its February 2020 level by early
  2021.
- Disposable income **rose** during the pandemic, even though unemployment reached 14.8 percent in April 2020 (the
  textbook's "15 percent"). After March 2020 it never fell below its February 2020 level again.
- Households saved much of the money. In the model the transfers raised $\bar{a}_c$, and the savings helped the strong
  recovery, and the inflation, of 2021–2022.
- **Data check.** The textbook's caption speaks of payments "totaling 3,000 dollars per person", but its text lists
  1,200 + 600 + 1,400 = 3,200 dollars.

**Your turn.** Saving is disposable income minus consumption. Compute it for every month and find the month in which it
was largest.

<details><summary>Hint</summary>

`income["saving"] = income.real_disp_income - income.real_consumption`, then `income.loc[income.saving.idxmax()]`.

</details>

In [81]:
# Your turn: try it here

## Figure 18: How the central bank sets the nominal interest rate

**Goal.** See where "money" comes in: the central bank sets the interest rate through the money market.

The **nominal interest rate is the opportunity cost of holding money**: cash and checking accounts pay nothing, while a
savings or money-market account pays $i$. At 1 percent you might as well hold cash; at 20 percent you would keep as little
as possible. So money demand $M^d$ falls as $i$ rises. The central bank decides how much money to supply, $M^s$: a
vertical line. The interest rate $i^*$ is where people want to hold exactly the money supplied. If $i$ were higher, people
would want less money than there is, and the rate would fall; if lower, the opposite.

As before, one function draws two textbook figures: this one (12.18) and a change in the money supply (12.19). Its arguments
are the new money supply `M_new` and the new level of money demand `m0_new` (demand is $M^d = m_0 - i$).

> **Python note.** `"Mᵈ′" if m0_new != M0 else "Mᵈ"` is a one-line `if`: it gives the first text when the condition is true
> and the second otherwise.

In [82]:
def money_experiment(M_new=M_BAR, m0_new=M0):
    """Figures 12.18 and 12.19: money supply and demand set the nominal rate; a change moves i* to i**."""
    i_star, i_new = market_rate(M_BAR), market_rate(M_new, m0_new)
    fig = go.Figure()
    if m0_new != M0:
        add_demand(fig, "Mᵈ", GRAY, M0, "dash")
    add_demand(fig, "Mᵈ′" if m0_new != M0 else "Mᵈ", BLUE, m0_new)
    if M_new != M_BAR:
        add_line(fig, [M_BAR, M_BAR], [0.5, 9.5], "Mˢ", GRAY, "dash")
        add_arrow(fig, M_BAR, 1.0, M_new, 1.0)                     # the change in the money supply
    add_line(fig, [M_new, M_new], [0.5, 9.5], "Mˢ′" if M_new != M_BAR else "Mˢ", TEAL)
    add_point(fig, M_BAR, i_star, "i*", 0, 0)
    if i_new != i_star:
        add_point(fig, M_new, i_new, "i**", 0, 0)
        add_arrow(fig, 0.5, i_star, 0.5, i_new)                    # the change in the interest rate
    set_axes(fig, X_MONEY, Y_NOMINAL)
    fig.update_layout(title=f"Money supply {M_new:.1f}, demand Mᵈ = {m0_new:.1f} − i  →  i = {i_star:.1f}% → {i_new:.1f}%",
                      margin=dict(r=90))
    return fig

In [83]:
interact(money_experiment,
         M_new=FloatSlider(value=5.0, min=3.0, max=7.0, step=0.5, description="money supply Mˢ", style=WIDE),
         m0_new=FloatSlider(value=10.0, min=8.0, max=12.0, step=0.5, description="money demand at i = 0, m₀", style=WIDE));

In [84]:
# @title Website version: Figure 18 with a precomputed slider for money demand
slider_figure(money_experiment, "m0_new", [8.0, 9.0, 10.0, 11.0, 12.0], start=10.0, label="money demand at i = 0, m₀")

**What to notice.**

- With a supply of 5 and demand $M^d = 10 - i$, the market clears at $i^* = 5$ percent.
- Move money demand: with the money supply fixed, every shift of demand moves the interest rate one for one. People need
  more money when prices or output are higher, or less when they pay by card, so demand shifts all the time.
- In the short run inflation is sticky, so the money market clears through the interest rate (and the velocity of money),
  not through the price level as in the quantity theory of Chapter 8.

**Your turn.** Money demand rises to $M^d = 11 - i$ because people make more purchases. What money supply keeps the interest
rate at 5 percent? Check it with `market_rate`.

<details><summary>Hint</summary>

You need $11 - i = M^s$ at $i = 5$. Then try `market_rate(6.0, m0=11.0)`.

</details>

In [85]:
# Your turn: try it here

## Figure 19: Raising the nominal interest rate

**Goal.** See what "tight money" means: to raise the interest rate, the central bank reduces the money supply.

The money supply shifts left, from $M^s$ to $M^{s\prime}$. At the old rate $i^*$, people now want more money than there is:
households ask their banks for cash, and banks must offer a higher interest rate on savings accounts to bring money demand
back in line with supply. The rate rises to $i^{**}$. We use our illustrative cut from 5 to 4 units of money.

In [86]:
interact(money_experiment,
         M_new=FloatSlider(value=4.0, min=3.0, max=7.0, step=0.5, description="new money supply Mˢ′", style=WIDE),
         m0_new=FloatSlider(value=10.0, min=8.0, max=12.0, step=0.5, description="money demand at i = 0, m₀", style=WIDE));

In [87]:
# @title Website version: Figure 19 with a precomputed slider for the new money supply
slider_figure(money_experiment, "M_new", [3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0], start=4.0, label="new money supply Mˢ′")

**What to notice.**

- Cutting the money supply from 5 to 4 raises the nominal rate from 5 to 6 percent: tight money means a higher interest rate.
- A larger money supply ("loose" money) lowers the rate: move the slider to the right.
- In practice the Fed changes the money supply by **open-market operations**: it sells government bonds to take money
  out of the economy, or buys them to put money in (see the key numbers in Figure 20).

**Your turn.** The Fed wants to cut the nominal rate from 5 to 4.5 percent. By how much must it increase the money supply?

<details><summary>Hint</summary>

With demand $10 - i$, the supply must equal $10 - 4.5$. Check with `market_rate(5.5)`.

</details>

In [88]:
# Your turn: try it here

## Figure 20: Targeting the nominal interest rate

**Goal.** See why central banks set an interest rate rather than a quantity of money.

Money demand shifts all the time (Figure 18). If the central bank kept the money supply fixed, every shift would move the
interest rate, and with it the real rate and output. Instead, modern central banks **peg** the interest rate: they supply
whatever money is demanded at the target rate $i^*$. The money supply curve becomes **horizontal** at $i^*$, and shifts in
money demand change the quantity of money, not the rate. Compare the two policies with the toggle.

> **Python note.** `ToggleButtons` shows one button per option, and the function receives the text of the chosen button.

In [89]:
def rate_target(m0_new=M0, i_target=5.0, policy="peg the interest rate"):
    """Figure 12.20: pegging i makes money supply horizontal; fixing M lets money-demand shocks move i."""
    fig = go.Figure()
    if m0_new != M0:
        add_demand(fig, "Mᵈ", GRAY, M0, "dash")
    add_demand(fig, "Mᵈ′" if m0_new != M0 else "Mᵈ", BLUE, m0_new)
    if policy == "peg the interest rate":
        i_eq, M_eq = i_target, money_demand(i_target, m0_new)
        add_line(fig, [0.5, 10.5], [i_target, i_target], "Mˢ", TEAL)
    else:
        M_eq = money_demand(i_target)                  # the supply that gave i_target before the shock
        i_eq = market_rate(M_eq, m0_new)
        add_line(fig, [M_eq, M_eq], [0.5, 9.5], "Mˢ", TEAL)
    add_point(fig, M_eq, i_eq, "", 0, 0)
    set_axes(fig, X_MONEY, Y_NOMINAL)
    fig.update_layout(title=f"Policy: {policy};  demand shift {m0_new - M0:+.1f}  →  i = {i_eq:.1f}%, M = {M_eq:.1f}",
                      margin=dict(r=90))
    return fig

In [90]:
interact(rate_target,
         m0_new=FloatSlider(value=11.0, min=8.0, max=12.0, step=0.5, description="money demand at i = 0, m₀", style=WIDE),
         i_target=FloatSlider(value=5.0, min=2.0, max=8.0, step=0.5, description="target rate i* (%)", style=WIDE),
         policy=ToggleButtons(options=["peg the interest rate", "fix the money supply"], value="peg the interest rate",
                              description="policy", style=WIDE));

In [91]:
# @title Website version: Figure 20 with a precomputed slider for money demand (the rate is pegged)
slider_figure(rate_target, "m0_new", [8.0, 9.0, 10.0, 11.0, 12.0], start=11.0, label="money demand at i = 0, m₀")

**What to notice.**

- With the rate pegged at 5 percent, a rise in money demand from $m_0 = 10$ to 11 is met by more money (6 instead of 5),
  and the interest rate does not move.
- Switch to "fix the money supply": the same shift raises the interest rate to 6 percent, which would push the real rate up
  and output down although nothing real happened. Pegging the rate shields the economy from money-demand shocks.
- **The IS/LM model.** An older version of the short-run model keeps the money supply fixed. Then higher output raises
  money demand and the interest rate, which gives an upward-sloping "LM" curve instead of our flat MP curve. Since central
  banks in fact adjust the money supply to keep the interest rate at its target, the two approaches lead to the same analysis.

### Key numbers: bond prices and interest rates

In an open-market operation the Fed buys or sells bonds, and the interest rate is hidden in the bond's price. A bond that
pays 100 dollars in one year and sells for 97 dollars today yields $3/97 = 3.1$ percent.

In [92]:
def bond_yield(price, face=100.0):
    """The one-year interest rate implied by a bond's price: (face - price) / price, in percent."""
    return 100 * (face - price) / price


for price in [95.0, 96.0, 97.0, 98.0, 99.0]:
    print(f"Price {price:.0f}  ->  yield {bond_yield(price):.2f}%")

Price 95  ->  yield 5.26%
Price 96  ->  yield 4.17%
Price 97  ->  yield 3.09%
Price 98  ->  yield 2.04%
Price 99  ->  yield 1.01%


When the Fed sells bonds, their price falls and the interest rate rises (tight money); when it buys bonds, their price rises
and the rate falls.

**Your turn.** At what price would a one-year bond with a face value of 100 dollars yield 5 percent?

<details><summary>Hint</summary>

Solve $(100 - P)/P = 0.05$, so $P = 100/1.05$. Check it with `bond_yield(100 / 1.05)`.

</details>

In [93]:
# Your turn: try it here

## Figure 21: The Great Inflation

**Goal.** Answer Exercise 12 of the textbook: how mistaking the productivity slowdown for a demand shock could have caused
the Great Inflation.

The four panels tell the story of Figure 13 with the model's diagrams:

- (a) Policymakers believe a negative demand shock $\bar{a}' < 0$ caused the slowdown: they think the IS curve shifted to IS′,
  and the economy is at $B$, below potential.
- (b) So they lower the real interest rate to $R' = \bar{r} + \bar{a}'/\bar{b}$, which, they believe, takes the economy back to
  potential at $C$.
- (c) In truth the IS curve never shifted. The lower rate moves the economy **down the true IS curve**, to $C$, above potential.
- (d) With output above potential, the Phillips curve raises inflation: $\downarrow R \Rightarrow \uparrow \tilde{Y} \Rightarrow \uparrow \pi$.

The figure has four panels, so we split the work: `perceived_panels` draws the two panels of what policymakers believe,
and `great_inflation` adds the truth and the readout.

In [94]:
def perceived_panels(fig, a_perc, R_new):
    """Figure 12.21 (a) and (b): the IS/MP diagram as policymakers see it."""
    for col in [1, 2]:
        add_is(fig, "IS", GRAY, dash="dash", row=1, col=col)
        add_is(fig, "IS′", TEAL, a_perc, row=1, col=col)
        add_point(fig, 0.0, R_BAR, "A", -8, 0, 1, col)
        add_point(fig, a_perc, R_BAR, "B", -8, 0, 1, col)
        set_axes(fig, X_OUTPUT, Y_RATE, 1, col)
    add_line(fig, MP_X, [R_BAR, R_BAR], "MP", BLUE, row=1, col=1)
    add_arrow(fig, 0.0, 0.6, a_perc, 0.6, 1, 1)
    add_line(fig, MP_X, [R_BAR, R_BAR], "MP", GRAY, "dash", 1, 2)
    add_line(fig, MP_X, [R_new, R_new], "MP′", BLUE, row=1, col=2)
    add_point(fig, 0.0, R_new, "C", -8, 0, 1, 2)
    add_arrow(fig, a_perc, 0.6, 0.0, 0.6, 1, 2)

In [95]:
def great_inflation(a_perc=-2.0, nu=NU):
    """Figure 12.21: the Fed sees a demand shock a' that never happened, cuts R, and inflation rises."""
    R_new = rate_for_output(0.0, a_perc)           # the cut that would close the perceived gap
    Y_true = is_curve(R_new)                       # but the true IS curve never moved
    dpi = phillips(Y_true, nu)
    fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.12, vertical_spacing=0.14,
                        subplot_titles=("(a) Policymakers see a fall in demand ...", "(b) so they lower R.",
                                        "(c) In truth, the IS curve never shifted ...", "(d) so ↓R ⇒ ↑Ỹ ⇒ ↑π."))
    perceived_panels(fig, a_perc, R_new)
    add_is(fig, "IS", TEAL, row=2, col=1)
    add_line(fig, MP_X, [R_BAR, R_BAR], "MP", GRAY, "dash", 2, 1)
    add_line(fig, MP_X, [R_new, R_new], "MP′", BLUE, row=2, col=1)
    add_point(fig, 0.0, R_BAR, "A", -8, 0, 2, 1)
    add_point(fig, Y_true, R_new, "C", -8, 0, 2, 1)
    add_arrow(fig, 0.0, 0.6, Y_true, 0.6, 2, 1)
    add_pc(fig, "Phillips curve", GREEN, nu, row=2, col=2)
    add_point(fig, 0.0, 0.0, "A", -8, -4, 2, 2)
    add_point(fig, Y_true, dpi, "B", -8, -4, 2, 2)
    set_axes(fig, X_OUTPUT, Y_RATE, 2, 1)
    set_axes(fig, X_OUTPUT, Y_DPI, 2, 2)
    fig.update_layout(height=820, title=f"Believed ā′ = {a_perc:.1f}  →  R′ = {R_new:.1f}%  →  true Ỹ = {Y_true:+.1f}%"
                                        f"  →  Δπ = {dpi:+.2f} a year")
    return fig

In [96]:
interact(great_inflation,
         a_perc=FloatSlider(value=-2.0, min=-3.0, max=0.0, step=0.5, description="demand shock policymakers believe in ā′", style=WIDE),
         nu=FloatSlider(value=0.5, min=0.25, max=1.0, step=0.05, description="slope ν̄", style=WIDE));

In [97]:
# @title Website version: Figure 21 with a precomputed slider for the believed demand shock
slider_figure(great_inflation, "a_perc", [-3.0, -2.5, -2.0, -1.5, -1.0, -0.5], start=-2.0,
              label="demand shock policymakers believe in ā′")

**What to notice.**

- Believing in a shock of $-2$, the Fed cuts $R$ from 5 to 4 percent. On the true IS curve this takes output to $+2$ percent,
  not 0, and inflation rises by 1 point a year for as long as the mistake lasts.
- The bigger the shock policymakers believe in, the bigger the cut, the bigger the boom, and the faster inflation rises.
- How can policymakers find out? Rising inflation with no other shock is the sign that output is above potential. In the
  1970s the oil shocks also raised inflation, so the mistake took years to see.

**Your turn.** Suppose the Fed recognizes its mistake and wants to stop inflation from rising. To what real interest rate
should it return? What if, in addition, it wants to bring inflation back down by 1 point a year?

<details><summary>Hint</summary>

Steady inflation needs $\tilde{Y} = 0$ on the true IS curve: `rate_for_output(0.0)`. Lowering inflation by 1 point a year
needs $\tilde{Y} = -1/\bar{\nu}$: try `rate_for_output(-2.0)`.

</details>

In [98]:
# Your turn: try it here

## Key numbers

This table collects the main numbers of the chapter. Each one is computed with the functions and data above.

In [99]:
volcker = simulate([6.0, 6.0, 6.0, 6.0, 6.0], pi_start=10.0)[1]
by_year = inflation.set_index("year")
rows = [
    ["Real rate when i = 7% and inflation = 2% (Fisher), %", real_rate(7.0, 2.0)],
    ["Y~ when the Fed raises R from 5% to 6%, %", is_curve(6.0)],
    ["R' that offsets a housing bust a' = -2, %", rate_for_output(0.0, a=-2.0)],
    ["Change in inflation at Y~ = -2% (nu = 0.5), points", phillips(-2.0)],
    ["Change in inflation at potential after an oil shock o = 1, points", phillips(0.0, o=1.0)],
    ["Volcker model: inflation after 5 years at Y~ = -2%, %", volcker[-1]],
    ["Fed funds rate, June 1981 peak, %", peak.fedfunds],
    ["Inflation 1980 (peak), %", by_year.inflation_cpi[1980]],
    ["Inflation 1983, %", by_year.inflation_cpi[1983]],
    ["Ex post real fed funds rate 1981, %", by_year.real_rate[1981]],
    ["10-year minus 1-month yield, April 3, 2023, points", book.yield_120m["2023-04-03"] - book.yield_1m["2023-04-03"]],
    ["Fall in real GDP, 2019Q4 to 2020Q2, %", fall],
    ["CPI inflation peak (June 2022), %", cpi.inflation_all.max()],
    ["Federal spending FY2020, % of GDP", outlays.outlays_gdp_pct.max()],
    ["Largest monthly jump in real disposable income (March 2021), trillions", jumps.iloc[0]],
    ["Nominal rate when the money supply falls from 5 to 4, %", market_rate(4.0)],
    ["Yield of a one-year bond bought at 97, %", bond_yield(97.0)],
]
pd.DataFrame(rows, columns=["Quantity", "Value"]).round(2)

,Quantity,Value
0,Real rate when i = 7% and inflation = 2% (Fish...,5.00
1,"Y~ when the Fed raises R from 5% to 6%, %",-2.00
2,"R' that offsets a housing bust a' = -2, %",4.00
3,"Change in inflation at Y~ = -2% (nu = 0.5), po...",-1.00
4,Change in inflation at potential after an oil ...,1.00
5,Volcker model: inflation after 5 years at Y~ =...,5.00
6,"Fed funds rate, June 1981 peak, %",19.10
7,"Inflation 1980 (peak), %",13.50
8,"Inflation 1983, %",3.16
9,"Ex post real fed funds rate 1981, %",6.00


## Summary

- The short-run model has three building blocks. Through the **MP curve**, the central bank's nominal interest rate sets the
  real interest rate, because inflation is **sticky**. Through the **IS curve**, the real rate sets short-run output. Through the
  **Phillips curve**, $\Delta\pi_t = \bar{\nu}\tilde{Y}_t + \bar{o}$, output changes inflation over time.
- In the IS/MP diagram, monetary policy can in principle offset demand shocks such as a housing bust, but long and variable
  lags make this hard in practice.
- Lowering inflation requires a temporary recession: the **Volcker disinflation** brought inflation from 13.5 percent in 1980 to
  3.2 percent in 1983, at the cost of the deepest recession in decades. Oil shocks, the belief in a permanent trade-off, and a
  productivity slowdown mistaken for a recession all contributed to the **Great Inflation** of the 1970s.
- Central banks set the interest rate by supplying whatever money is demanded at their target. Through the term structure,
  the short rate also moves longer-term rates.
- The next chapter adds the central bank's **policy rule** and expectations, leading to the AS/AD framework.